# Notebook 09 — Development and frozen input preparation
Historical preparation workflow. Outputs are cleared; saved development results are in `results/reranking_eval_v1`. Final inference and metrics are in `09_cross_encoder_reranking.ipynb`. Do not run both notebooks against the same checkpoint directory.


In [ ]:
# CELL 1 — Setup & Inspect Retrieval Benchmark Artifacts

from pathlib import Path
import subprocess
import json
import pandas as pd
import torch

# ------------------------------------
# 1. Environment
# ------------------------------------

print("ENVIRONMENT")
print("-" * 55)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# ------------------------------------
# 2. Clone GitHub repository
# ------------------------------------

REPO_URL = (
    "https://github.com/"
    "P-Tilemachos/Greek-News-NLP-API.git"
)

REPO_DIR = Path("/content/Greek-News-NLP-API")

if not REPO_DIR.exists():

    print("\nCloning GitHub repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

else:
    assert (REPO_DIR / ".git").is_dir()
    print("\nRepository already exists.")

print("Repository:", REPO_DIR)

# ------------------------------------
# 3. Locate Notebook 08 results
# ------------------------------------

EVAL_DIR = (
    REPO_DIR / "results" / "retrieval_eval_v2"
)

required_files = [
    "queries.jsonl",
    "manifest.json",
    "evaluation_predictions.csv",
    "evaluation_metrics.csv",
    "evaluation_manifest.json",
]

print("\nRETRIEVAL ARTIFACTS")
print("-" * 55)

for filename in required_files:

    path = EVAL_DIR / filename

    status = "FOUND" if path.is_file() else "MISSING"

    print(f"{filename}: {status}")

    assert path.is_file(), f"Missing file: {path}"

# ------------------------------------
# 4. Load frozen queries
# ------------------------------------

with (
    EVAL_DIR / "queries.jsonl"
).open(encoding="utf-8") as f:

    queries = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(queries) == 600

print("\nFROZEN QUERIES")
print("-" * 55)

print("Total queries:", len(queries))
print("Query fields:", list(queries[0].keys()))

# ------------------------------------
# 5. Load retrieval predictions
# ------------------------------------

predictions_df = pd.read_csv(
    EVAL_DIR / "evaluation_predictions.csv"
)

print("\nRETRIEVAL PREDICTIONS")
print("-" * 55)

print("Rows:", len(predictions_df))
print("Columns:", predictions_df.columns.tolist())

# ------------------------------------
# 6. Load existing evaluation metrics
# ------------------------------------

metrics_df = pd.read_csv(
    EVAL_DIR / "evaluation_metrics.csv"
)

print("\nEXISTING RETRIEVAL METRICS")
print("-" * 55)

display(metrics_df)

# ------------------------------------
# 7. Load evaluation manifest
# ------------------------------------

evaluation_manifest = json.loads(
    (
        EVAL_DIR / "evaluation_manifest.json"
    ).read_text(encoding="utf-8")
)

print("\nEVALUATION MANIFEST")
print("-" * 55)

print(
    "Fields:",
    list(evaluation_manifest.keys())
)

# ------------------------------------
# 8. Final verification
# ------------------------------------

print("\n" + "=" * 55)
print("NOTEBOOK 09 — INITIAL SETUP")
print("=" * 55)

print("Repository: VERIFIED")
print("Notebook 08 artifacts: VERIFIED")
print("Frozen queries:", len(queries))
print("Existing results: LOADED")

print("\nOK — Notebook 09 Cell 1 completed")

In [ ]:
# CELL 2 — Inspect Retrieval Pipeline & Candidate Sources

import json
from pathlib import Path

# ------------------------------------
# 1. Verify existing variables
# ------------------------------------

assert REPO_DIR.is_dir()
assert EVAL_DIR.is_dir()
assert len(queries) == 600

print("NOTEBOOK 09 — RETRIEVAL INSPECTION")
print("=" * 60)

# ------------------------------------
# 2. Inspect evaluation configuration
# ------------------------------------

for section in [
    "dataset",
    "bm25",
    "dense_retrieval",
    "hybrid",
]:

    value = evaluation_manifest.get(section, {})

    print(f"\nSECTION: {section.upper()}")
    print("-" * 55)

    print(
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )[:3000]
    )

# ------------------------------------
# 3. Inspect frozen query partitions
# ------------------------------------

print("\nQUERY PARTITIONS")
print("-" * 55)

partition_counts = {}

for record in queries:

    key = (
        record["partition"],
        record["query_type"],
    )

    partition_counts[key] = (
        partition_counts.get(key, 0) + 1
    )

for key, count in sorted(partition_counts.items()):
    print(f"{key}: {count}")

# ------------------------------------
# 4. Inspect prediction structure
# ------------------------------------

print("\nPREDICTION STRUCTURE")
print("-" * 55)

print("Columns:", predictions_df.columns.tolist())

print("\nRows per retrieval method:")

print(
    predictions_df["method"].value_counts().to_string()
)

candidate_columns = [
    col
    for col in predictions_df.columns
    if any(
        term in col.lower()
        for term in [
            "candidate",
            "retrieved",
            "document_ids",
            "top_k",
        ]
    )
]

print("\nCandidate-list columns:", candidate_columns)

# ------------------------------------
# 5. Inspect available retrieval files
# ------------------------------------

print("\nRELEVANT REPOSITORY FILES")
print("-" * 55)

search_dirs = [
    REPO_DIR / "notebooks",
    REPO_DIR / "results",
    REPO_DIR / "src",
    REPO_DIR / "data",
]

for directory in search_dirs:

    if not directory.is_dir():
        continue

    matches = []

    for path in directory.rglob("*"):

        if not path.is_file():
            continue

        name = path.name.lower()

        if any(
            term in name
            for term in [
                "retriev",
                "hybrid",
                "index",
                "corpus",
                "article",
                "rerank",
                "bm25",
            ]
        ):
            matches.append(
                str(path.relative_to(REPO_DIR))
            )

    print(f"\n{directory.name}/")

    for match in matches[:25]:
        print(" ", match)

    if len(matches) > 25:
        print(f"  ... and {len(matches) - 25} more")

# ------------------------------------
# 6. Inspect Notebook 08 structure
# ------------------------------------

notebook08_path = (
    REPO_DIR / "notebooks" / "08_hybrid_retrieval.ipynb"
)

print("\nNOTEBOOK 08")
print("-" * 55)

if notebook08_path.is_file():

    notebook08 = json.loads(
        notebook08_path.read_text(encoding="utf-8")
    )

    print("Notebook found:", notebook08_path.name)
    print("Total cells:", len(notebook08["cells"]))

    keywords = [
        "rrf",
        "candidate",
        "bm25",
        "faiss",
        "retrieve",
    ]

    for index, cell in enumerate(
        notebook08["cells"],
        start=1,
    ):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        matches = [
            word
            for word in keywords
            if word in source.lower()
        ]

        if len(matches) >= 2:

            print(
                f"Cell {index}: "
                f"{', '.join(matches)}"
            )

else:

    print("Notebook 08 not found at expected path.")

# ------------------------------------
# 7. Final status
# ------------------------------------

print("\n" + "=" * 60)

if candidate_columns:
    print("Candidate columns detected.")
else:
    print(
        "Full candidate rankings are not present "
        "in the evaluation predictions CSV."
    )

print("\nOK — Retrieval sources inspected")

In [ ]:
# CELL 3 — Inspect Original Hybrid Retrieval Implementation

import json

# ------------------------------------
# 1. Load original Notebook 08
# ------------------------------------

NOTEBOOK08_PATH = (
    REPO_DIR / "notebooks" / "08_hybrid_retrieval.ipynb"
)

assert NOTEBOOK08_PATH.is_file()

with NOTEBOOK08_PATH.open(encoding="utf-8") as f:
    notebook08 = json.load(f)

cells = notebook08["cells"]

print("NOTEBOOK 08 — CODE INSPECTION")
print("=" * 60)

print("Total cells:", len(cells))

# ------------------------------------
# 2. Inspect relevant original cells
# ------------------------------------

target_cells = [4, 8, 9, 13]

for cell_number in target_cells:

    cell = cells[cell_number - 1]

    assert cell["cell_type"] == "code"

    source = "".join(cell.get("source", []))

    print("\n" + "=" * 60)
    print(f"ORIGINAL NOTEBOOK 08 — CELL {cell_number}")
    print("=" * 60)

    print(source[:7000])

    if len(source) > 7000:
        print("\n[TRUNCATED — cell exceeds 7000 characters]")

# ------------------------------------
# 3. Inspect relevant function names
# ------------------------------------

print("\n" + "=" * 60)
print("FUNCTION DEFINITIONS")
print("=" * 60)

keywords = [
    "bm25",
    "rrf",
    "dense",
    "faiss",
    "hybrid",
    "retriev",
    "candidate",
    "corpus",
]

for cell_number, cell in enumerate(cells, start=1):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    for line in source.splitlines():

        stripped = line.strip()

        if stripped.startswith("def "):

            if any(
                word in stripped.lower()
                for word in keywords
            ):
                print(
                    f"Cell {cell_number}: {stripped}"
                )

print("\nOK — Original retrieval code inspected")

In [ ]:
# CELL 4 — Inspect Exact Retrieval Function Implementations

import re

assert "notebook08" in globals()

target_functions = [
    "search_bm25",
    "search_dense",
    "search_hybrid",
    "fuse_rankings",
]

found = set()

for cell_number, cell in enumerate(
    notebook08["cells"], start=1
):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    for name in target_functions:

        pattern = rf"^def\s+{name}\s*\("

        for start, line in enumerate(lines):

            if not re.match(pattern, line):
                continue

            found.add(name)

            # Find the end of this function.
            end = start + 1

            while end < len(lines):
                current = lines[end]

                if (
                    current.strip()
                    and not current[0].isspace()
                    and not current.startswith("#")
                ):
                    break

                end += 1

            function_lines = lines[start:end]

            print("\n" + "=" * 60)
            print(f"NOTEBOOK 08 — CELL {cell_number}")
            print(f"FUNCTION: {name}")
            print("=" * 60)

            print("\n".join(function_lines[:70]))

            if len(function_lines) > 70:
                print(
                    f"\n[TRUNCATED: "
                    f"{len(function_lines) - 70} more lines]"
                )

print("\n" + "=" * 60)
print("FUNCTION VERIFICATION")
print("=" * 60)

for name in target_functions:
    print(
        f"{name}:",
        "FOUND" if name in found else "MISSING"
    )

assert set(target_functions) == found

print("\nOK — Retrieval implementations inspected")

In [ ]:
# CELL 5 — Inspect Corpus and Retrieval Setup

import json

assert "notebook08" in globals()

print("NOTEBOOK 08 — DATA & INDEX SETUP")
print("=" * 60)

# ------------------------------------
# 1. Inspect original setup cells
# ------------------------------------

target_cells = [1, 2, 3, 4, 5, 6, 7, 8]

keywords = [
    "jsonl",
    "docs",
    "bm25",
    "tokenize_greek",
    "dense_index",
    "dense_row_to_id",
    "doc_by_id",
    "embedder",
    "faiss",
    "huggingface",
    "hf_hub",
    "data_dir",
    "drive.mount",
]

for cell_number in target_cells:

    cell = notebook08["cells"][cell_number - 1]

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    found_keywords = [
        word
        for word in keywords
        if word in source.lower()
    ]

    if not found_keywords:
        continue

    print("\n" + "=" * 60)
    print(f"NOTEBOOK 08 — CELL {cell_number}")
    print("Keywords:", ", ".join(found_keywords))
    print("=" * 60)

    print(source[:4500])

    if len(source) > 4500:
        print("\n[TRUNCATED]")

# ------------------------------------
# 2. Inspect complete Hybrid sorting
# ------------------------------------

hybrid_source = "".join(
    notebook08["cells"][8].get("source", [])
)

print("\n" + "=" * 60)
print("HYBRID RETRIEVAL — FINAL SORTING")
print("=" * 60)

print(
    "\n".join(
        hybrid_source.splitlines()[-35:]
    )
)

# ------------------------------------
# 3. Final verification
# ------------------------------------

print("\nOK — Corpus and retrieval setup inspected")

In [ ]:
# CELL 6 — Load and Verify Original Article Corpus

from google.colab import drive
from pathlib import Path
import json
import hashlib

# ------------------------------------
# 1. Mount Google Drive
# ------------------------------------

drive.mount("/content/drive")

DATA_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp"
)

assert DATA_DIR.is_dir()

# ------------------------------------
# 2. Define original dataset files
# ------------------------------------

DATASET_FILES = {
    "train": DATA_DIR / "el_train.jsonl",
    "val": DATA_DIR / "el_val.jsonl",
    "test": DATA_DIR / "el_test.jsonl",
}

print("DATASET FILES")
print("-" * 55)

for split, path in DATASET_FILES.items():

    assert path.is_file(), f"Missing: {path}"

    print(f"{split}: FOUND")

# ------------------------------------
# 3. Verify SHA256 hashes
# ------------------------------------

def file_sha256(path):

    hasher = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


expected_hashes = evaluation_manifest[
    "dataset"
]["source_file_sha256"]

print("\nDATASET INTEGRITY")
print("-" * 55)

for split, path in DATASET_FILES.items():

    actual_hash = file_sha256(path)
    expected_hash = expected_hashes[split]

    assert actual_hash == expected_hash, (
        f"SHA256 mismatch for {split}"
    )

    print(f"{split}: SHA256 VERIFIED")

# ------------------------------------
# 4. Load article records
# ------------------------------------

corpus_records = []
split_counts = {}

for split, path in DATASET_FILES.items():

    records = []

    with path.open(encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)
            records.append(record)

    split_counts[split] = len(records)
    corpus_records.extend(records)

print("\nCORPUS SIZE")
print("-" * 55)

for split, count in split_counts.items():
    print(f"{split}: {count}")

print("Total articles:", len(corpus_records))

# ------------------------------------
# 5. Verify article schema
# ------------------------------------

required_fields = {
    "document_id",
    "text_clean",
    "GPT-IPTC-label",
}

for record in corpus_records:

    assert required_fields.issubset(record)

    assert isinstance(
        record["document_id"], str
    )

    assert isinstance(
        record["text_clean"], str
    )

    assert record["text_clean"].strip()

# ------------------------------------
# 6. Verify unique document IDs
# ------------------------------------

corpus_ids = [
    record["document_id"]
    for record in corpus_records
]

assert len(corpus_records) == 5250
assert len(set(corpus_ids)) == 5250

print("\nDOCUMENT VERIFICATION")
print("-" * 55)

print("Unique document IDs:", len(set(corpus_ids)))
print("Duplicate IDs: 0")
print("Document fields:", list(corpus_records[0].keys()))

# ------------------------------------
# 7. Verify frozen query sources
# ------------------------------------

corpus_id_set = set(corpus_ids)

assert all(
    query["source_id"] in corpus_id_set
    for query in queries
)

print("\nFROZEN QUERY VERIFICATION")
print("-" * 55)

print("Frozen queries:", len(queries))
print("All source articles found: YES")

# ------------------------------------
# 8. Final verification
# ------------------------------------

print("\n" + "=" * 55)
print("NOTEBOOK 09 — CORPUS VERIFIED")
print("=" * 55)

print("Total articles:", len(corpus_records))
print("Dataset hashes: VERIFIED")
print("Document IDs: UNIQUE")
print("Query source IDs: VERIFIED")

print("\nOK — Original article corpus loaded")

In [ ]:
# CELL 7 — Prepare Corpus and Inspect Original BM25 Tokenization

import re

# ------------------------------------
# 1. Verify previous cells
# ------------------------------------

assert "corpus_records" in globals(), (
    "Run Cell 6 first."
)

assert "notebook08" in globals(), (
    "Run Cell 3 first."
)

assert len(corpus_records) == 5250
assert len(queries) == 600

print("NOTEBOOK 09 — CORPUS PREPARATION")
print("=" * 60)

# ------------------------------------
# 2. Prepare document mapping
# ------------------------------------

docs09 = [
    {
        "id": record["document_id"],
        "text": record["text_clean"],
        "label": record["GPT-IPTC-label"],
    }
    for record in corpus_records
]

doc_by_id09 = {
    doc["id"]: doc
    for doc in docs09
}

assert len(docs09) == 5250
assert len(doc_by_id09) == 5250

assert all(
    isinstance(doc["text"], str)
    and doc["text"].strip()
    for doc in docs09
)

print("\nDOCUMENT MAPPING")
print("-" * 55)

print("Documents:", len(docs09))
print("Unique document IDs:", len(doc_by_id09))
print("Document fields:", list(docs09[0].keys()))

# ------------------------------------
# 3. Verify frozen query references
# ------------------------------------

missing_source_ids = [
    q["source_id"]
    for q in queries
    if q["source_id"] not in doc_by_id09
]

assert not missing_source_ids

print("\nQUERY SOURCE VERIFICATION")
print("-" * 55)

print("Frozen queries:", len(queries))
print("Missing source IDs:", len(missing_source_ids))

# ------------------------------------
# 4. Inspect original Greek tokenizer
# ------------------------------------

print("\nORIGINAL BM25 TOKENIZER")
print("=" * 60)

tokenizer_found = False

for cell_number, cell in enumerate(
    notebook08["cells"], start=1
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    for start, line in enumerate(lines):

        if not re.match(
            r"^def\s+tokenize_greek\s*\(",
            line
        ):
            continue

        tokenizer_found = True

        end = start + 1

        while end < len(lines):

            current = lines[end]

            if (
                current.strip()
                and not current[0].isspace()
                and not current.startswith("#")
            ):
                break

            end += 1

        print("Original cell:", cell_number)
        print("\n".join(lines[start:end]))

assert tokenizer_found, (
    "Original tokenize_greek function not found."
)

# ------------------------------------
# 5. Inspect original BM25 setup
# ------------------------------------

print("\nORIGINAL BM25 CONSTRUCTION")
print("=" * 60)

setup_found = False

patterns = [
    r"\bBM25Okapi\s*\(",
    r"\bbm25_full\s*=",
]

for cell_number, cell in enumerate(
    notebook08["cells"], start=1
):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(
        cell.get("source", [])
    ).splitlines()

    for index, line in enumerate(lines):

        if not any(
            re.search(pattern, line)
            for pattern in patterns
        ):
            continue

        setup_found = True

        start = max(0, index - 8)
        end = min(len(lines), index + 10)

        print(f"\nNotebook 08 — Cell {cell_number}")

        print("\n".join(lines[start:end]))

assert setup_found, (
    "Original BM25 construction not found."
)

# ------------------------------------
# 6. Final checks
# ------------------------------------

print("\n" + "=" * 60)
print("CELL 7 VERIFICATION")
print("=" * 60)

print("Corpus mapping: VERIFIED")
print("Frozen query sources: VERIFIED")
print("Original tokenizer: FOUND")
print("Original BM25 setup: FOUND")

print("\nOK — Corpus ready for BM25 reconstruction")

In [ ]:
# CELL 8 — Reconstruct Original BM25 (CPU)

%pip install -q rank-bm25==0.2.2

import re
import unicodedata
import numpy as np

from rank_bm25 import BM25Okapi

# ------------------------------------
# 1. Verify corpus
# ------------------------------------

assert len(docs09) == 5250
assert len(doc_by_id09) == 5250
assert len(queries) == 600

print("RECONSTRUCTING ORIGINAL BM25")
print("=" * 60)

# ------------------------------------
# 2. Original Notebook 08 tokenizer
# ------------------------------------

def tokenize_greek09(text):

    text = unicodedata.normalize(
        "NFD",
        text.casefold()
    )

    text = "".join(
        char
        for char in text
        if unicodedata.category(char) != "Mn"
    )

    return re.findall(
        r"[^\W_]+",
        text,
        flags=re.UNICODE
    )


test_tokens = tokenize_greek09(
    "Η Αθήνα αντιμετωπίζει ΠΛΗΜΜΥΡΕΣ."
)

assert test_tokens == [
    "η",
    "αθηνα",
    "αντιμετωπιζει",
    "πλημμυρεσ"
]

print("\nTokenizer: VERIFIED")

# ------------------------------------
# 3. Tokenize original articles
# ------------------------------------

print("\nTokenizing 5,250 articles...")

tokenized_docs09 = [
    tokenize_greek09(doc["text"])
    for doc in docs09
]

assert len(tokenized_docs09) == 5250
assert all(tokenized_docs09)

print("Tokenization: COMPLETED")

# ------------------------------------
# 4. Build BM25 index
# ------------------------------------

print("\nBuilding BM25 index...")

bm25_full09 = BM25Okapi(
    tokenized_docs09,
    k1=1.5,
    b=0.75,
    epsilon=0.25
)

assert bm25_full09.corpus_size == 5250

print("BM25 index: CREATED")

# ------------------------------------
# 5. Original BM25 search function
# ------------------------------------

def search_bm2509(query, k=5):

    tokens = tokenize_greek09(query)

    if not tokens or k <= 0:
        return []

    scores = bm25_full09.get_scores(tokens)

    order = np.argsort(
        -scores,
        kind="stable"
    )

    results = []

    for i in order:

        if scores[i] <= 0:
            continue

        doc = docs09[int(i)]

        results.append({
            "id": doc["id"],
            "score": float(scores[i]),
            "label": doc["label"],
            "snippet": doc["text"][:350],
        })

        if len(results) == k:
            break

    return results

# ------------------------------------
# 6. Test frozen evaluation query
# ------------------------------------

sample_query = next(
    q for q in queries
    if q["partition"] == "evaluation"
    and q["query_type"] == "late"
)

print("\nFROZEN QUERY TEST")
print("-" * 55)

print("Query ID:", sample_query["query_id"])
print("Gold document:", sample_query["source_id"])

bm25_results09 = search_bm2509(
    sample_query["query"],
    k=10
)

assert len(bm25_results09) > 0

print("\nTOP-5 BM25 RESULTS")

for rank, result in enumerate(
    bm25_results09[:5],
    start=1
):

    print(
        f"{rank}. {result['id']} "
        f"| score={result['score']:.4f}"
    )

# ------------------------------------
# 7. Verify against Notebook 08
# ------------------------------------

reference = predictions_df[
    (
        predictions_df["query_id"]
        == sample_query["query_id"]
    )
    &
    (
        predictions_df["method"]
        == "BM25 Full"
    )
]

assert len(reference) == 1

expected_rank = int(
    reference.iloc[0]["rank"]
)

actual_rank = next(
    (
        rank
        for rank, result in enumerate(
            bm25_results09,
            start=1
        )
        if result["id"] == sample_query["source_id"]
    ),
    None
)

print("\nREPRODUCIBILITY CHECK")
print("-" * 55)

print("Notebook 08 gold rank:", expected_rank)
print("Notebook 09 gold rank:", actual_rank)

assert actual_rank == expected_rank, (
    "BM25 ranking differs from Notebook 08."
)

# ------------------------------------
# 8. Final verification
# ------------------------------------

print("\n" + "=" * 60)
print("CELL 8 VERIFICATION")
print("=" * 60)

print("Documents:", len(docs09))
print("BM25 index: READY")
print("Original parameters: VERIFIED")
print("Frozen query result: MATCHED")
print("Runtime: CPU")

print("\nOK — Original BM25 reconstructed")

In [ ]:
# CELL 9 — Load Existing FAISS Index & E5 Model (CPU)

%pip install -q faiss-cpu huggingface-hub sentence-transformers

import json
import hashlib
import numpy as np
import faiss
import torch

from pathlib import Path
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

# ------------------------------------
# 1. Verify previous steps
# ------------------------------------

assert len(docs09) == 5250
assert len(doc_by_id09) == 5250
assert bm25_full09.corpus_size == 5250

print("LOADING ORIGINAL DENSE RETRIEVAL")
print("=" * 60)

# ------------------------------------
# 2. Original index configuration
# ------------------------------------

dense_config = evaluation_manifest["dense_retrieval"]

INDEX_REPO = dense_config["index_repo"]
INDEX_REVISION = dense_config["index_revision"]

INDEX_DIR = DATA_DIR / "rag_index"

expected_hashes = dense_config["index_sha256"]

print("\nINDEX CONFIGURATION")
print("-" * 55)

print("Repository:", INDEX_REPO)
print("Revision:", INDEX_REVISION)

# ------------------------------------
# 3. SHA256 helper
# ------------------------------------

def sha256_file09(path):

    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            hasher.update(chunk)

    return hasher.hexdigest()

# ------------------------------------
# 4. Locate verified index files
# ------------------------------------

index_files09 = {}

for filename in ["articles.faiss", "docs.json"]:

    local_path = INDEX_DIR / filename
    expected_sha = expected_hashes[filename]

    if (
        local_path.is_file()
        and sha256_file09(local_path) == expected_sha
    ):

        selected_path = local_path
        print("Verified local file:", filename)

    else:

        print("Downloading pinned file:", filename)

        selected_path = Path(
            hf_hub_download(
                repo_id=INDEX_REPO,
                filename=filename,
                repo_type="dataset",
                revision=INDEX_REVISION,
            )
        )

    actual_sha = sha256_file09(selected_path)

    assert actual_sha == expected_sha, (
        f"SHA256 mismatch: {filename}"
    )

    index_files09[filename] = selected_path

    print("SHA256 VERIFIED:", filename)

# ------------------------------------
# 5. Load FAISS index
# ------------------------------------

print("\nLoading FAISS index...")

dense_index09 = faiss.read_index(
    str(index_files09["articles.faiss"])
)

dense_docs09 = json.loads(
    index_files09["docs.json"].read_text(
        encoding="utf-8"
    )
)

assert dense_index09.ntotal == 5250
assert dense_index09.d == 384
assert dense_index09.metric_type == faiss.METRIC_INNER_PRODUCT

assert len(dense_docs09) == 5250

print("\nFAISS INDEX")
print("-" * 55)

print("Vectors:", dense_index09.ntotal)
print("Dimensions:", dense_index09.d)
print("Metric: Inner Product")

# ------------------------------------
# 6. Verify document mapping
# ------------------------------------

dense_row_to_id09 = [
    doc["id"]
    for doc in dense_docs09
]

assert len(set(dense_row_to_id09)) == 5250

assert set(dense_row_to_id09) == set(doc_by_id09)

for dense_doc in dense_docs09:

    doc_id = dense_doc["id"]

    assert dense_doc["text"] == (
        doc_by_id09[doc_id]["text"]
    )

print("\nDOCUMENT VERIFICATION")
print("-" * 55)

print("Matching document IDs: YES")
print("Matching article texts: YES")
print("FAISS row mapping: VERIFIED")

# ------------------------------------
# 7. Load original E5 model
# ------------------------------------

E5_MODEL_ID = dense_config["model"]

print("\nLoading multilingual E5 on CPU...")
print("Model:", E5_MODEL_ID)

embedder09 = SentenceTransformer(
    E5_MODEL_ID,
    device="cpu",
)

embedder09.max_seq_length = 256

assert embedder09.max_seq_length == 256

print("E5 model loaded.")

# ------------------------------------
# 8. Test frozen query embedding
# ------------------------------------

sample_query_text = sample_query["query"]

query_embedding09 = embedder09.encode(
    ["query: " + sample_query_text.strip()],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

query_embedding09 = np.ascontiguousarray(
    query_embedding09,
    dtype=np.float32
)

assert query_embedding09.shape == (1, 384)

# ------------------------------------
# 9. Test FAISS search
# ------------------------------------

scores09, indices09 = dense_index09.search(
    query_embedding09,
    5
)

print("\nDENSE RETRIEVAL SMOKE TEST")
print("-" * 55)

print("Query ID:", sample_query["query_id"])
print("Gold document:", sample_query["source_id"])

for rank, (score, index) in enumerate(
    zip(scores09[0], indices09[0]),
    start=1,
):

    assert index >= 0

    doc_id = dense_row_to_id09[int(index)]

    print(
        f"{rank}. {doc_id} "
        f"| score={float(score):.4f}"
    )

# ------------------------------------
# 10. Final verification
# ------------------------------------

print("\n" + "=" * 60)
print("CELL 9 VERIFICATION")
print("=" * 60)

print("FAISS vectors:", dense_index09.ntotal)
print("Embedding dimensions:", dense_index09.d)
print("Dataset mapping: VERIFIED")
print("Index SHA256: VERIFIED")
print("E5 model: LOADED")
print("Dense retrieval: WORKING")
print("Runtime: CPU")

print("\nOK — FAISS and E5 loaded successfully")

In [ ]:
# ============================================================
# CELL 10 — Reconstruct Hybrid RRF and Verify Rankings
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU
# ============================================================

# 1. Install required dependency
%pip install -q rank-bm25==0.2.2

import re
import unicodedata
import numpy as np

from rank_bm25 import BM25Okapi


print("CELL 10 — HYBRID RRF RECONSTRUCTION")
print("=" * 60)


# ------------------------------------------------------------
# 2. Verify required variables
# ------------------------------------------------------------

required_variables = [
    "docs09",
    "dense_index09",
    "embedder09",
    "dense_row_to_id09",
    "evaluation_manifest",
    "predictions_df",
    "queries",
]

missing = [
    name
    for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing variables: "
        + ", ".join(missing)
        + "\nRun the required setup Cells 1, 3, 6, 7, 8, 9."
    )

assert len(docs09) == 5250
assert dense_index09.ntotal == 5250
assert dense_index09.d == 384
assert len(dense_row_to_id09) == 5250

print("\nRequired variables: VERIFIED")


# ------------------------------------------------------------
# 3. Original Greek tokenizer
# ------------------------------------------------------------

def tokenize_greek09(text):

    text = unicodedata.normalize(
        "NFD",
        text.casefold()
    )

    text = "".join(
        char
        for char in text
        if unicodedata.category(char) != "Mn"
    )

    return re.findall(
        r"[^\W_]+",
        text,
        flags=re.UNICODE
    )


assert tokenize_greek09("ΠΛΗΜΜΥΡΕΣ") == [
    "πλημμυρεσ"
]

print("Greek tokenizer: READY")


# ------------------------------------------------------------
# 4. Restore BM25 if missing
# ------------------------------------------------------------

if "bm25_full09" not in globals():

    print("\nRebuilding BM25 index...")

    tokenized_docs09 = [
        tokenize_greek09(doc["text"])
        for doc in docs09
    ]

    assert len(tokenized_docs09) == 5250
    assert all(tokenized_docs09)

    bm25_full09 = BM25Okapi(
        tokenized_docs09,
        k1=1.5,
        b=0.75,
        epsilon=0.25
    )

    print("BM25 index rebuilt.")

else:

    print("Existing BM25 index found.")

assert bm25_full09.corpus_size == 5250


# ------------------------------------------------------------
# 5. BM25 Full search
# ------------------------------------------------------------

def search_bm2509(query, k=5):

    tokens = tokenize_greek09(query)

    if not tokens or k <= 0:
        return []

    scores = bm25_full09.get_scores(tokens)

    order = np.argsort(
        -scores,
        kind="stable"
    )

    results = []

    for i in order:

        if scores[i] <= 0:
            continue

        doc = docs09[int(i)]

        results.append({
            "id": doc["id"],
            "score": float(scores[i]),
            "label": doc["label"],
            "snippet": doc["text"][:350],
        })

        if len(results) == k:
            break

    return results


print("BM25 Full: READY")


# ------------------------------------------------------------
# 6. Original Hybrid configuration
# ------------------------------------------------------------

hybrid_config09 = evaluation_manifest["hybrid"]

CANDIDATE_K09 = int(
    hybrid_config09["candidate_k"]
)

RRF_K09 = int(
    hybrid_config09["rrf_k"]
)

TOP_K09 = int(
    hybrid_config09["top_k"]
)

assert CANDIDATE_K09 == 50
assert RRF_K09 == 60
assert TOP_K09 == 10

print("\nHYBRID CONFIGURATION")
print("-" * 55)

print("BM25 candidates:", CANDIDATE_K09)
print("Dense candidates:", CANDIDATE_K09)
print("RRF k:", RRF_K09)
print("Final Top-K:", TOP_K09)


# ------------------------------------------------------------
# 7. Dense E5 search
# ------------------------------------------------------------

def search_dense09(query, k=50):

    if not query.strip() or k <= 0:
        return []

    query_text = "query: " + query.strip()

    embedding = embedder09.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    embedding = np.ascontiguousarray(
        embedding,
        dtype=np.float32
    )

    assert embedding.shape == (
        1,
        dense_index09.d
    )

    scores, indices = dense_index09.search(
        embedding,
        min(k, dense_index09.ntotal)
    )

    results = []

    for score, row_idx in zip(
        scores[0],
        indices[0]
    ):

        if row_idx < 0:
            continue

        doc_id = dense_row_to_id09[
            int(row_idx)
        ]

        results.append({
            "id": doc_id,
            "score": float(score),
        })

    return results


print("Dense E5: READY")


# ------------------------------------------------------------
# 8. Hybrid Reciprocal Rank Fusion
# ------------------------------------------------------------

def search_hybrid09(query, k=10):

    if not query.strip() or k <= 0:
        return []

    # Fixed pool: BM25 Top-50 + Dense Top-50.
    # At most 100 unique candidate documents.

    bm25_results = search_bm2509(
        query,
        k=CANDIDATE_K09
    )

    dense_results = search_dense09(
        query,
        k=CANDIDATE_K09
    )

    bm25_ranks = {
        row["id"]: rank
        for rank, row in enumerate(
            bm25_results,
            start=1
        )
    }

    dense_ranks = {
        row["id"]: rank
        for rank, row in enumerate(
            dense_results,
            start=1
        )
    }

    candidate_ids = (
        set(bm25_ranks)
        | set(dense_ranks)
    )

    fused = []

    for doc_id in candidate_ids:

        b_rank = bm25_ranks.get(doc_id)
        d_rank = dense_ranks.get(doc_id)

        rrf_score = 0.0

        if b_rank is not None:
            rrf_score += (
                1.0 / (RRF_K09 + b_rank)
            )

        if d_rank is not None:
            rrf_score += (
                1.0 / (RRF_K09 + d_rank)
            )

        best_rank = min(
            rank
            for rank in (b_rank, d_rank)
            if rank is not None
        )

        fused.append({
            "id": doc_id,
            "score": rrf_score,
            "bm25_rank": b_rank,
            "dense_rank": d_rank,
            "best_rank": best_rank,
        })

    # Deterministic tie-breaking from Notebook 08
    fused.sort(
        key=lambda row: (
            -row["score"],
            row["best_rank"],
            row["id"],
        )
    )

    return fused[:k]


print("Hybrid RRF: READY")


# ------------------------------------------------------------
# 9. Retrieval evaluation metrics
# ------------------------------------------------------------

def retrieval_metrics09(results, gold_id):

    ranked_ids = [
        row["id"]
        for row in results
    ]

    if gold_id in ranked_ids:
        rank = ranked_ids.index(gold_id) + 1
    else:
        rank = None

    return {
        "recall@1": float(
            rank is not None and rank <= 1
        ),
        "recall@5": float(
            rank is not None and rank <= 5
        ),
        "recall@10": float(
            rank is not None and rank <= 10
        ),
        "mrr@10": (
            1.0 / rank
            if rank is not None and rank <= 10
            else 0.0
        ),
    }


# ------------------------------------------------------------
# 10. Select frozen evaluation queries
# ------------------------------------------------------------

evaluation_queries09 = [
    q for q in queries
    if q["partition"] == "evaluation"
]

lead_queries09 = [
    q for q in evaluation_queries09
    if q["query_type"] == "lead"
]

late_queries09 = [
    q for q in evaluation_queries09
    if q["query_type"] == "late"
]

assert len(lead_queries09) == 240
assert len(late_queries09) == 240

smoke_queries09 = (
    lead_queries09[:2]
    + late_queries09[:2]
)

print("\nFROZEN EVALUATION QUERIES")
print("-" * 55)

print("Lead queries:", len(lead_queries09))
print("Late queries:", len(late_queries09))
print("Smoke-test queries:", len(smoke_queries09))


# ------------------------------------------------------------
# 11. Verify Hybrid RRF against Notebook 08
# ------------------------------------------------------------

print("\nHYBRID RRF REPRODUCIBILITY TEST")
print("=" * 60)

mismatches09 = []

metric_names09 = [
    "recall@1",
    "recall@5",
    "recall@10",
    "mrr@10",
]

for q in smoke_queries09:

    results = search_hybrid09(
        q["query"],
        k=TOP_K09
    )

    actual = retrieval_metrics09(
        results,
        q["source_id"]
    )

    reference = predictions_df[
        (
            predictions_df["query_id"]
            == q["query_id"]
        )
        &
        (
            predictions_df["method"]
            == "Hybrid RRF"
        )
    ]

    assert len(reference) == 1

    expected = reference.iloc[0]

    matched = all(
        np.isclose(
            actual[metric],
            float(expected[metric]),
            atol=1e-8,
            rtol=0
        )
        for metric in metric_names09
    )

    print(
        q["query_id"],
        "|",
        "MATCH" if matched else "MISMATCH",
        "| R@1:",
        actual["recall@1"],
        "| R@10:",
        actual["recall@10"]
    )

    if not matched:
        mismatches09.append(q["query_id"])


# ------------------------------------------------------------
# 12. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CELL 10 VERIFICATION")
print("=" * 60)

print("BM25: READY")
print("Dense E5: READY")
print("Hybrid RRF: READY")
print("Queries checked:", len(smoke_queries09))
print("Mismatches:", len(mismatches09))

if mismatches09:

    print("\nWARNING — Reproducibility mismatch.")
    print("Mismatched query IDs:", mismatches09)
    print("Do not proceed to Cell 11.")

else:

    print("\nOK — Hybrid RRF reconstructed and verified")

In [ ]:
# CELL 11 — Verify Hybrid Candidate Pool (CPU)

import pandas as pd

# ------------------------------------
# 1. Verify Cell 10
# ------------------------------------

assert "search_hybrid09" in globals()
assert "mismatches09" in globals()
assert len(mismatches09) == 0

assert len(queries) == 600
assert CANDIDATE_K09 == 50
assert RRF_K09 == 60

print("HYBRID CANDIDATE POOL VERIFICATION")
print("=" * 60)

# ------------------------------------
# 2. Select late development queries
# ------------------------------------

dev_late_queries09 = [
    q for q in queries
    if q["partition"] == "dev"
    and q["query_type"] == "late"
]

assert len(dev_late_queries09) == 60

preview_queries09 = dev_late_queries09[:10]

print("\nQUERY CONFIGURATION")
print("-" * 55)

print("Total late development queries:", 60)
print("Preview queries:", len(preview_queries09))
print("BM25 candidate limit:", CANDIDATE_K09)
print("Dense candidate limit:", CANDIDATE_K09)

# ------------------------------------
# 3. Retrieve real Hybrid candidates
# ------------------------------------

preview_rows09 = []

for i, query_record in enumerate(
    preview_queries09,
    start=1
):

    candidates = search_hybrid09(
        query_record["query"],
        k=100,
    )

    candidate_ids = [
        result["id"]
        for result in candidates
    ]

    assert len(candidate_ids) == len(set(candidate_ids))
    assert 0 < len(candidate_ids) <= 100

    assert all(
        doc_id in doc_by_id09
        for doc_id in candidate_ids
    )

    gold_id = query_record["source_id"]

    gold_rank = (
        candidate_ids.index(gold_id) + 1
        if gold_id in candidate_ids
        else None
    )

    preview_rows09.append({
        "query_id": query_record["query_id"],
        "candidate_count": len(candidate_ids),
        "gold_rank": gold_rank,
        "gold_in_top1": (
            gold_rank is not None
            and gold_rank <= 1
        ),
        "gold_in_top5": (
            gold_rank is not None
            and gold_rank <= 5
        ),
        "gold_in_top10": (
            gold_rank is not None
            and gold_rank <= 10
        ),
        "gold_in_top20": (
            gold_rank is not None
            and gold_rank <= 20
        ),
        "gold_in_top50": (
            gold_rank is not None
            and gold_rank <= 50
        ),
        "gold_in_pool": gold_rank is not None,
    })

    print(
        f"[{i}/10] {query_record['query_id']}"
        f" | candidates={len(candidate_ids)}"
        f" | gold_rank={gold_rank}"
    )

# ------------------------------------
# 4. Build preview dataframe
# ------------------------------------

dev_preview_df09 = pd.DataFrame(
    preview_rows09
)

assert len(dev_preview_df09) == 10
assert dev_preview_df09["query_id"].is_unique

# ------------------------------------
# 5. Candidate coverage metrics
# ------------------------------------

print("\nCANDIDATE COVERAGE")
print("-" * 55)

coverage_columns = {
    "gold_in_top1": "Recall@1",
    "gold_in_top5": "Recall@5",
    "gold_in_top10": "Recall@10",
    "gold_in_top20": "Recall@20",
    "gold_in_top50": "Recall@50",
    "gold_in_pool": "Candidate Pool Recall",
}

for column, label in coverage_columns.items():

    value = float(
        dev_preview_df09[column].mean()
    )

    print(f"{label}: {value:.4f}")

# ------------------------------------
# 6. Display query-level results
# ------------------------------------

print("\nDEVELOPMENT QUERY PREVIEW")

display(dev_preview_df09)

# ------------------------------------
# 7. Final verification
# ------------------------------------

print("\n" + "=" * 60)
print("CELL 11 VERIFICATION")
print("=" * 60)

print("Hybrid RRF: WORKING")
print("Late development queries tested:", len(dev_preview_df09))
print("Candidate pool: BM25 Top50 + Dense Top50")
print("Maximum unique candidates: 100")
print("Original benchmark files: UNCHANGED")

print("\nOK — Hybrid candidate pool verified")

In [ ]:
# CELL 12 — Prepare and Save Cross-Encoder Candidates (CPU)

import json
import hashlib
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Verify previous steps
# --------------------------------------------------

assert "search_hybrid09" in globals()
assert "doc_by_id09" in globals()
assert "dev_late_queries09" in globals()

assert len(dev_late_queries09) == 60
assert len(doc_by_id09) == 5250
assert CANDIDATE_K09 == 50
assert RRF_K09 == 60

print("PREPARING CROSS-ENCODER CANDIDATES")
print("=" * 60)

# --------------------------------------------------
# 2. Define output directory
# --------------------------------------------------

RERANK_DIR09 = (
    DATA_DIR / "results" / "reranking_eval_v1"
)

RERANK_DIR09.mkdir(
    parents=True,
    exist_ok=True
)

CANDIDATES_PATH09 = (
    RERANK_DIR09 / "dev_late_candidates.jsonl"
)

# --------------------------------------------------
# 3. Retrieve candidates for all 60 queries
# --------------------------------------------------

dev_candidate_pools09 = []
candidate_summary09 = []

print("\nRetrieving Hybrid RRF candidates...")

for i, q in enumerate(
    dev_late_queries09,
    start=1
):

    results = search_hybrid09(
        q["query"],
        k=100
    )

    assert 0 < len(results) <= 100

    candidate_ids = [
        row["id"]
        for row in results
    ]

    assert len(candidate_ids) == len(set(candidate_ids))

    assert all(
        doc_id in doc_by_id09
        for doc_id in candidate_ids
    )

    # Preserve exact Hybrid ranking and scores.
    candidates = []

    for rank, row in enumerate(results, start=1):

        candidates.append({
            "rank": rank,
            "document_id": row["id"],
            "rrf_score": float(row["score"]),
            "bm25_rank": row["bm25_rank"],
            "dense_rank": row["dense_rank"],
        })

    gold_id = q["source_id"]

    gold_rank = next(
        (
            row["rank"]
            for row in candidates
            if row["document_id"] == gold_id
        ),
        None
    )

    dev_candidate_pools09.append({
        "query_id": q["query_id"],
        "partition": "dev",
        "query_type": "late",
        "query": q["query"],
        "source_id": gold_id,
        "candidates": candidates,
    })

    candidate_summary09.append({
        "query_id": q["query_id"],
        "candidate_count": len(candidates),
        "gold_rank": gold_rank,
        "gold_in_top1": (
            gold_rank is not None and gold_rank <= 1
        ),
        "gold_in_top5": (
            gold_rank is not None and gold_rank <= 5
        ),
        "gold_in_top10": (
            gold_rank is not None and gold_rank <= 10
        ),
        "gold_in_top20": (
            gold_rank is not None and gold_rank <= 20
        ),
        "gold_in_pool": gold_rank is not None,
    })

    if i % 10 == 0:
        print(f"Processed {i}/60 queries")

# --------------------------------------------------
# 4. Validate candidate pools
# --------------------------------------------------

assert len(dev_candidate_pools09) == 60

assert len({
    q["query_id"]
    for q in dev_candidate_pools09
}) == 60

dev_candidate_summary09 = pd.DataFrame(
    candidate_summary09
)

assert len(dev_candidate_summary09) == 60

# Verify consistency with Cell 11 preview.
for _, previous in dev_preview_df09.iterrows():

    current = dev_candidate_summary09[
        dev_candidate_summary09["query_id"]
        == previous["query_id"]
    ]

    assert len(current) == 1

    current = current.iloc[0]

    assert int(current["candidate_count"]) == int(
        previous["candidate_count"]
    )

    assert current["gold_rank"] == previous["gold_rank"]

print("\nAll 60 candidate pools verified.")

# --------------------------------------------------
# 5. Calculate development coverage
# --------------------------------------------------

print("\nFULL LATE DEVELOPMENT COVERAGE")
print("-" * 55)

for column, label in [
    ("gold_in_top1", "Recall@1"),
    ("gold_in_top5", "Recall@5"),
    ("gold_in_top10", "Recall@10"),
    ("gold_in_top20", "Recall@20"),
    ("gold_in_pool", "Candidate Pool Recall"),
]:

    value = float(
        dev_candidate_summary09[column].mean()
    )

    print(f"{label}: {value:.4f}")

print(
    "\nAverage candidate count:",
    round(
        dev_candidate_summary09[
            "candidate_count"
        ].mean(),
        2
    )
)

print(
    "Total query-document pairs:",
    int(
        dev_candidate_summary09[
            "candidate_count"
        ].sum()
    )
)

# --------------------------------------------------
# 6. Save frozen candidate pools
# --------------------------------------------------

jsonl_text = "".join(
    json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
    ) + "\n"
    for record in dev_candidate_pools09
)

jsonl_bytes = jsonl_text.encode("utf-8")

candidates_sha256 = hashlib.sha256(
    jsonl_bytes
).hexdigest()

if CANDIDATES_PATH09.exists():

    assert (
        CANDIDATES_PATH09.read_bytes()
        == jsonl_bytes
    ), (
        "Existing candidate file differs. "
        "Do not overwrite before investigating."
    )

    print("\nExisting candidate file verified.")

else:

    CANDIDATES_PATH09.write_bytes(
        jsonl_bytes
    )

    print("\nCandidate file saved.")

assert (
    CANDIDATES_PATH09.read_bytes()
    == jsonl_bytes
)

# --------------------------------------------------
# 7. Final report
# --------------------------------------------------

print("\n" + "=" * 60)
print("CELL 12 VERIFICATION")
print("=" * 60)

print("Development queries:", len(dev_candidate_pools09))
print("Query type: late")
print("Candidate source: Hybrid RRF")
print("BM25 candidate limit:", CANDIDATE_K09)
print("Dense candidate limit:", CANDIDATE_K09)
print("Original evaluation files: UNCHANGED")

print("\nCandidate SHA256:", candidates_sha256)

print("\nSaved to:")
print(CANDIDATES_PATH09)

print("\nOK — Cross-Encoder candidates prepared")

In [ ]:
# CELL 13 — Load Multilingual Cross-Encoder and Smoke Test
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU

import time
import hashlib
import numpy as np
import torch

from pathlib import Path
from huggingface_hub import HfApi
from sentence_transformers import CrossEncoder

# --------------------------------------------------
# 1. Verify previous steps
# --------------------------------------------------

assert "dev_candidate_pools09" in globals()
assert "doc_by_id09" in globals()
assert "CANDIDATES_PATH09" in globals()

assert len(dev_candidate_pools09) == 60
assert len(doc_by_id09) == 5250
assert CANDIDATES_PATH09.is_file()

EXPECTED_SHA256 = (
    "f95253f6fa83388e28880f40e21c2eac379d9b394ae412510ab1f199d7ccb057"
)

actual_sha256 = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

assert actual_sha256 == EXPECTED_SHA256

print("Candidate dataset: VERIFIED")
print("Queries:", len(dev_candidate_pools09))

# --------------------------------------------------
# 2. Cross-Encoder configuration
# --------------------------------------------------

RERANK_MODEL_ID09 = "BAAI/bge-reranker-v2-m3"

RERANK_DEVICE09 = "cpu"
RERANK_MAX_LENGTH09 = 512
RERANK_BATCH_SIZE09 = 2

print("\nCROSS-ENCODER CONFIGURATION")
print("-" * 55)

print("Model:", RERANK_MODEL_ID09)
print("Device:", RERANK_DEVICE09)
print("Max sequence length:", RERANK_MAX_LENGTH09)
print("Batch size:", RERANK_BATCH_SIZE09)

# --------------------------------------------------
# 3. Resolve exact model revision
# --------------------------------------------------

print("\nResolving model revision...")

RERANK_REVISION09 = HfApi().model_info(
    repo_id=RERANK_MODEL_ID09
).sha

assert RERANK_REVISION09

print("Revision:", RERANK_REVISION09)

# --------------------------------------------------
# 4. Load Cross-Encoder on CPU
# --------------------------------------------------

print("\nLoading multilingual Cross-Encoder...")
print("First download may take several minutes.")

load_start = time.perf_counter()

reranker09 = CrossEncoder(
    RERANK_MODEL_ID09,
    revision=RERANK_REVISION09,
    device=RERANK_DEVICE09,
    max_length=RERANK_MAX_LENGTH09,
    trust_remote_code=False,
)

load_seconds = time.perf_counter() - load_start

print("\nMODEL LOADED")
print("-" * 55)

print("Loading time:", round(load_seconds, 2), "seconds")
print("Device:", RERANK_DEVICE09)
print("Model revision:", RERANK_REVISION09)

# --------------------------------------------------
# 5. Select one real development query
# --------------------------------------------------

sample_pool09 = dev_candidate_pools09[0]

sample_query09 = sample_pool09["query"]

# Use only the first 3 actual Hybrid candidates.
sample_candidates09 = sample_pool09["candidates"][:3]

assert len(sample_candidates09) == 3

pairs09 = []

for candidate in sample_candidates09:

    doc_id = candidate["document_id"]

    assert doc_id in doc_by_id09

    article_text = doc_by_id09[doc_id]["text"]

    pairs09.append((
        sample_query09,
        article_text,
    ))

print("\nSMOKE TEST INPUT")
print("-" * 55)

print("Query ID:", sample_pool09["query_id"])
print("Candidate pairs:", len(pairs09))
print("Gold document:", sample_pool09["source_id"])

# --------------------------------------------------
# 6. Cross-Encoder inference
# --------------------------------------------------

print("\nRunning Cross-Encoder inference...")

inference_start = time.perf_counter()

scores09 = reranker09.predict(
    pairs09,
    batch_size=RERANK_BATCH_SIZE09,
    show_progress_bar=False,
    convert_to_numpy=True,
)

inference_seconds = (
    time.perf_counter() - inference_start
)

scores09 = np.asarray(
    scores09,
    dtype=float
).reshape(-1)

assert scores09.shape == (3,)
assert np.isfinite(scores09).all()

print("Inference completed.")
print("Elapsed seconds:", round(inference_seconds, 3))

# --------------------------------------------------
# 7. Inspect candidate scores
# --------------------------------------------------

print("\nHYBRID VS CROSS-ENCODER")
print("-" * 55)

scored_candidates09 = []

for candidate, score in zip(
    sample_candidates09,
    scores09
):

    scored_candidates09.append({
        "document_id": candidate["document_id"],
        "hybrid_rank": candidate["rank"],
        "reranker_score": float(score),
    })

# Higher scores indicate greater predicted relevance.
scored_candidates09.sort(
    key=lambda row: (
        -row["reranker_score"],
        row["hybrid_rank"],
        row["document_id"],
    )
)

for new_rank, row in enumerate(
    scored_candidates09,
    start=1,
):

    is_gold = (
        row["document_id"]
        == sample_pool09["source_id"]
    )

    print(
        f"{new_rank}. {row['document_id']}"
        f" | Hybrid rank={row['hybrid_rank']}"
        f" | Reranker score={row['reranker_score']:.4f}"
        f" | Gold={is_gold}"
    )

# --------------------------------------------------
# 8. Final verification
# --------------------------------------------------

assert len(scored_candidates09) == 3

assert len({
    row["document_id"]
    for row in scored_candidates09
}) == 3

print("\n" + "=" * 60)
print("CELL 13 VERIFICATION")
print("=" * 60)

print("Cross-Encoder: LOADED")
print("Runtime: CPU")
print("Candidate pairs tested:", len(pairs09))
print("Scores: VALID")
print("Model revision: RECORDED")
print("Original retrieval files: UNCHANGED")

print("\nOK — Cross-Encoder smoke test completed")

In [ ]:
# CELL 14 — Focused Cross-Encoder Reranking Test (CPU)

import time
import numpy as np
import pandas as pd

# ------------------------------------
# 1. Verify previous cells
# ------------------------------------

assert "reranker09" in globals()
assert "dev_candidate_pools09" in globals()
assert "doc_by_id09" in globals()

assert RERANK_MAX_LENGTH09 == 512

print("FOCUSED CROSS-ENCODER TEST")
print("=" * 60)

# ------------------------------------
# 2. Select a development query
# ------------------------------------

target_query_id = "late_005"
TOP_K_TEST09 = 5

pool14 = next(
    record
    for record in dev_candidate_pools09
    if record["query_id"] == target_query_id
)

query14 = pool14["query"]
gold_id14 = pool14["source_id"]

candidates14 = pool14["candidates"][:TOP_K_TEST09]

assert len(candidates14) == TOP_K_TEST09

hybrid_ids14 = [
    row["document_id"]
    for row in candidates14
]

hybrid_gold_rank14 = (
    hybrid_ids14.index(gold_id14) + 1
    if gold_id14 in hybrid_ids14
    else None
)

print("\nQUERY INFORMATION")
print("-" * 55)

print("Query ID:", target_query_id)
print("Gold document:", gold_id14)
print("Hybrid gold rank:", hybrid_gold_rank14)
print("Candidates:", len(candidates14))

# ------------------------------------
# 3. Prepare query-document pairs
# ------------------------------------

pairs14 = []
token_audit14 = []

for candidate in candidates14:

    doc_id = candidate["document_id"]

    assert doc_id in doc_by_id09

    article_text = doc_by_id09[doc_id]["text"]

    pairs14.append((query14, article_text))

    # Inspect length before truncation.
    full_encoding = reranker09.tokenizer(
        query14,
        article_text,
        truncation=False,
    )

    full_token_count = len(
        full_encoding["input_ids"]
    )

    token_audit14.append({
        "document_id": doc_id,
        "full_pair_tokens": full_token_count,
        "exceeds_512": (
            full_token_count > RERANK_MAX_LENGTH09
        ),
    })

token_audit_df14 = pd.DataFrame(token_audit14)

print("\nINPUT LENGTH AUDIT")
print("-" * 55)

print(
    "Pairs exceeding 512 tokens:",
    int(token_audit_df14["exceeds_512"].sum()),
    "/",
    len(token_audit_df14),
)

print(
    "Maximum full pair tokens:",
    token_audit_df14["full_pair_tokens"].max()
)

# ------------------------------------
# 4. Run Cross-Encoder
# ------------------------------------

print("\nRunning reranking on 5 pairs...")

start14 = time.perf_counter()

scores14 = reranker09.predict(
    pairs14,
    batch_size=RERANK_BATCH_SIZE09,
    show_progress_bar=False,
    convert_to_numpy=True,
)

elapsed14 = time.perf_counter() - start14

scores14 = np.asarray(
    scores14,
    dtype=float,
).reshape(-1)

assert scores14.shape == (TOP_K_TEST09,)
assert np.isfinite(scores14).all()

print("Inference time:", round(elapsed14, 3), "seconds")

# ------------------------------------
# 5. Rerank candidates
# ------------------------------------

reranked14 = []

for candidate, score in zip(candidates14, scores14):

    reranked14.append({
        "document_id": candidate["document_id"],
        "hybrid_rank": int(candidate["rank"]),
        "reranker_score": float(score),
        "is_gold": (
            candidate["document_id"] == gold_id14
        ),
    })

reranked14.sort(
    key=lambda row: (
        -row["reranker_score"],
        row["hybrid_rank"],
        row["document_id"],
    )
)

reranker_gold_rank14 = next(
    (
        rank
        for rank, row in enumerate(reranked14, start=1)
        if row["is_gold"]
    ),
    None
)

# ------------------------------------
# 6. Compare rankings
# ------------------------------------

print("\nRERANKED TOP-5")
print("-" * 55)

for rank, row in enumerate(reranked14, start=1):

    marker = " <-- GOLD" if row["is_gold"] else ""

    print(
        f"{rank}. {row['document_id']}"
        f" | Hybrid rank={row['hybrid_rank']}"
        f" | Score={row['reranker_score']:.4f}"
        f"{marker}"
    )

print("\nRANK COMPARISON")
print("-" * 55)

print("Hybrid gold rank:", hybrid_gold_rank14)
print("Cross-Encoder gold rank:", reranker_gold_rank14)

if reranker_gold_rank14 < hybrid_gold_rank14:
    print("Result: IMPROVED")
elif reranker_gold_rank14 == hybrid_gold_rank14:
    print("Result: UNCHANGED")
else:
    print("Result: DEGRADED")

# ------------------------------------
# 7. Final verification
# ------------------------------------

assert len(reranked14) == TOP_K_TEST09
assert len(set(hybrid_ids14)) == TOP_K_TEST09

assert hybrid_gold_rank14 is not None
assert reranker_gold_rank14 is not None

print("\n" + "=" * 60)
print("CELL 14 VERIFICATION")
print("=" * 60)

print("Development query:", target_query_id)
print("Cross-Encoder: WORKING")
print("Candidates reranked:", TOP_K_TEST09)
print("Gold ranking: VERIFIED")
print("Original evaluation files: UNCHANGED")

print("\nOK — Focused reranking test completed")

In [ ]:
# CELL 15 — Cross-Encoder Development Pilot (CPU)
# Notebook: 09_cross_encoder_reranking.ipynb

import time
import numpy as np
import pandas as pd

# ------------------------------------
# 1. Verify previous steps
# ------------------------------------

assert "reranker09" in globals()
assert "dev_candidate_pools09" in globals()
assert "doc_by_id09" in globals()

assert len(dev_candidate_pools09) == 60
assert RERANK_MAX_LENGTH09 == 512

PILOT_QUERIES09 = 5
PILOT_TOP_K09 = 5

print("CROSS-ENCODER DEVELOPMENT PILOT")
print("=" * 60)

print("Queries:", PILOT_QUERIES09)
print("Candidates per query:", PILOT_TOP_K09)
print("Maximum pairs:", PILOT_QUERIES09 * PILOT_TOP_K09)
print("Device: CPU")

# ------------------------------------
# 2. Select fixed development subset
# ------------------------------------

pilot_queries09 = dev_candidate_pools09[
    :PILOT_QUERIES09
]

assert len(pilot_queries09) == 5
assert all(
    q["partition"] == "dev"
    and q["query_type"] == "late"
    for q in pilot_queries09
)

# ------------------------------------
# 3. Run paired reranking
# ------------------------------------

pilot_rows09 = []
pilot_predictions09 = []

total_start09 = time.perf_counter()

for i, query_record in enumerate(
    pilot_queries09,
    start=1
):

    query_id = query_record["query_id"]
    query_text = query_record["query"]
    gold_id = query_record["source_id"]

    candidates = query_record["candidates"][
        :PILOT_TOP_K09
    ]

    assert len(candidates) == PILOT_TOP_K09

    hybrid_ids = [
        c["document_id"]
        for c in candidates
    ]

    assert len(set(hybrid_ids)) == PILOT_TOP_K09

    hybrid_gold_rank = (
        hybrid_ids.index(gold_id) + 1
        if gold_id in hybrid_ids
        else None
    )

    # Build real query-document pairs.
    pairs = [
        (
            query_text,
            doc_by_id09[c["document_id"]]["text"]
        )
        for c in candidates
    ]

    print(
        f"\n[{i}/{PILOT_QUERIES09}] {query_id}",
        flush=True
    )

    # --------------------------------
    # Cross-Encoder inference
    # --------------------------------

    start = time.perf_counter()

    scores = reranker09.predict(
        pairs,
        batch_size=RERANK_BATCH_SIZE09,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    elapsed = time.perf_counter() - start

    scores = np.asarray(
        scores,
        dtype=float
    ).reshape(-1)

    assert scores.shape == (PILOT_TOP_K09,)
    assert np.isfinite(scores).all()

    # --------------------------------
    # Deterministic reranking
    # --------------------------------

    scored = [
        {
            "document_id": candidate["document_id"],
            "hybrid_rank": int(candidate["rank"]),
            "reranker_score": float(score),
        }
        for candidate, score in zip(
            candidates,
            scores
        )
    ]

    scored.sort(
        key=lambda x: (
            -x["reranker_score"],
            x["hybrid_rank"],
            x["document_id"],
        )
    )

    reranked_ids = [
        row["document_id"]
        for row in scored
    ]

    reranker_gold_rank = (
        reranked_ids.index(gold_id) + 1
        if gold_id in reranked_ids
        else None
    )

    # --------------------------------
    # Evaluate gold article ranking
    # --------------------------------

    if hybrid_gold_rank is None:
        hybrid_rr = 0.0
    else:
        hybrid_rr = 1.0 / hybrid_gold_rank

    if reranker_gold_rank is None:
        reranker_rr = 0.0
    else:
        reranker_rr = 1.0 / reranker_gold_rank

    if reranker_rr > hybrid_rr:
        outcome = "IMPROVED"
    elif reranker_rr < hybrid_rr:
        outcome = "DEGRADED"
    else:
        outcome = "UNCHANGED"

    pilot_rows09.append({
        "query_id": query_id,
        "hybrid_gold_rank": hybrid_gold_rank,
        "reranker_gold_rank": reranker_gold_rank,
        "hybrid_rr": hybrid_rr,
        "reranker_rr": reranker_rr,
        "outcome": outcome,
        "inference_seconds": elapsed,
    })

    pilot_predictions09.append({
        "query_id": query_id,
        "source_id": gold_id,
        "hybrid_ids": hybrid_ids,
        "reranked_candidates": scored,
    })

    print(
        f"Hybrid rank: {hybrid_gold_rank}"
        f" | Reranker rank: {reranker_gold_rank}"
        f" | {outcome}"
    )

    print(
        f"Inference: {elapsed:.2f} seconds",
        flush=True
    )

# ------------------------------------
# 4. Build comparison DataFrame
# ------------------------------------

pilot_df09 = pd.DataFrame(
    pilot_rows09
)

assert len(pilot_df09) == PILOT_QUERIES09
assert pilot_df09["query_id"].is_unique

print("\nQUERY-LEVEL RESULTS")
print("-" * 60)

display(pilot_df09.round(4))

# ------------------------------------
# 5. Aggregate retrieval metrics
# ------------------------------------

hybrid_r1 = (
    pilot_df09["hybrid_gold_rank"] == 1
).mean()

reranker_r1 = (
    pilot_df09["reranker_gold_rank"] == 1
).mean()

hybrid_r5 = (
    pilot_df09["hybrid_gold_rank"].notna()
).mean()

reranker_r5 = (
    pilot_df09["reranker_gold_rank"].notna()
).mean()

hybrid_mrr = pilot_df09["hybrid_rr"].mean()
reranker_mrr = pilot_df09["reranker_rr"].mean()

print("\nPILOT METRICS")
print("=" * 60)

print(
    f"Hybrid Recall@1:       {hybrid_r1:.4f}"
)
print(
    f"Cross-Encoder Recall@1: {reranker_r1:.4f}"
)

print(
    f"Hybrid Recall@5:       {hybrid_r5:.4f}"
)
print(
    f"Cross-Encoder Recall@5: {reranker_r5:.4f}"
)

print(
    f"Hybrid MRR@5:          {hybrid_mrr:.4f}"
)
print(
    f"Cross-Encoder MRR@5:    {reranker_mrr:.4f}"
)

print("\nRANKING OUTCOMES")
print("-" * 60)

print(
    pilot_df09["outcome"]
    .value_counts()
    .to_string()
)

# ------------------------------------
# 6. Timing
# ------------------------------------

total_elapsed09 = (
    time.perf_counter() - total_start09
)

print("\nPERFORMANCE")
print("-" * 60)

print(
    f"Total pilot time: {total_elapsed09:.2f} sec"
)

print(
    "Average inference time per query:",
    f"{pilot_df09['inference_seconds'].mean():.2f} sec"
)

# ------------------------------------
# 7. Final verification
# ------------------------------------

assert len(pilot_predictions09) == 5

assert all(
    len(row["reranked_candidates"]) == 5
    for row in pilot_predictions09
)

print("\n" + "=" * 60)
print("CELL 15 VERIFICATION")
print("=" * 60)

print("Development queries:", len(pilot_df09))
print("Candidates per query:", PILOT_TOP_K09)
print("Cross-Encoder: WORKING")
print("Paired ranking comparison: COMPLETED")
print("Evaluation queries: NOT USED")
print("Original benchmark files: UNCHANGED")

print("\nOK — Cross-Encoder development pilot completed")

In [ ]:
# CELL 16 — Save Cross-Encoder Development Pilot Results
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU

import json
import hashlib
import sys
import numpy as np
import pandas as pd

from pathlib import Path
from importlib import metadata

# --------------------------------------------------
# 1. Verify previous cells
# --------------------------------------------------

assert "pilot_df09" in globals()
assert "pilot_predictions09" in globals()
assert "dev_candidate_pools09" in globals()
assert "RERANK_REVISION09" in globals()
assert "CANDIDATES_PATH09" in globals()

assert len(pilot_df09) == 5
assert len(pilot_predictions09) == 5

assert pilot_df09["query_id"].is_unique

assert PILOT_QUERIES09 == 5
assert PILOT_TOP_K09 == 5

assert RERANK_MAX_LENGTH09 == 512
assert RERANK_DEVICE09 == "cpu"

print("CROSS-ENCODER PILOT — SAVE RESULTS")
print("=" * 60)

# --------------------------------------------------
# 2. Verify query IDs and original candidates
# --------------------------------------------------

expected_queries = dev_candidate_pools09[:5]

expected_ids = [
    q["query_id"]
    for q in expected_queries
]

assert pilot_df09["query_id"].tolist() == expected_ids

assert [
    p["query_id"]
    for p in pilot_predictions09
] == expected_ids

for original, prediction in zip(
    expected_queries,
    pilot_predictions09
):

    assert original["query_id"] == prediction["query_id"]

    assert original["source_id"] == prediction["source_id"]

    original_top5 = [
        candidate["document_id"]
        for candidate in original["candidates"][:5]
    ]

    assert prediction["hybrid_ids"] == original_top5

    reranked_ids = [
        row["document_id"]
        for row in prediction["reranked_candidates"]
    ]

    assert len(reranked_ids) == 5
    assert set(reranked_ids) == set(original_top5)

    scores = [
        row["reranker_score"]
        for row in prediction["reranked_candidates"]
    ]

    assert np.isfinite(scores).all()

print("Pilot queries: VERIFIED")
print("Original candidate sets: VERIFIED")
print("Reranked predictions: VERIFIED")

# --------------------------------------------------
# 3. Define output directory
# --------------------------------------------------

SAVE_DIR09 = (
    DATA_DIR / "results" / "reranking_eval_v1"
)

SAVE_DIR09.mkdir(
    parents=True,
    exist_ok=True
)

PILOT_PREDICTIONS_PATH09 = (
    SAVE_DIR09 / "pilot_top5_predictions.jsonl"
)

PILOT_METRICS_PATH09 = (
    SAVE_DIR09 / "pilot_top5_metrics.csv"
)

PILOT_MANIFEST_PATH09 = (
    SAVE_DIR09 / "pilot_top5_manifest.json"
)

# --------------------------------------------------
# 4. Prepare prediction file
# --------------------------------------------------

predictions_text = "".join(
    json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
    for record in pilot_predictions09
)

predictions_bytes = predictions_text.encode("utf-8")

predictions_sha256 = hashlib.sha256(
    predictions_bytes
).hexdigest()

# --------------------------------------------------
# 5. Prepare metrics CSV
# --------------------------------------------------

metrics_bytes = pilot_df09.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.8f",
).encode("utf-8")

metrics_sha256 = hashlib.sha256(
    metrics_bytes
).hexdigest()

# --------------------------------------------------
# 6. Calculate measured results
# --------------------------------------------------

hybrid_r1 = float(
    (pilot_df09["hybrid_gold_rank"] == 1).mean()
)

reranker_r1 = float(
    (pilot_df09["reranker_gold_rank"] == 1).mean()
)

hybrid_r5 = float(
    pilot_df09["hybrid_gold_rank"].notna().mean()
)

reranker_r5 = float(
    pilot_df09["reranker_gold_rank"].notna().mean()
)

hybrid_mrr5 = float(
    pilot_df09["hybrid_rr"].mean()
)

reranker_mrr5 = float(
    pilot_df09["reranker_rr"].mean()
)

outcome_counts = {
    name: int(
        (pilot_df09["outcome"] == name).sum()
    )
    for name in [
        "IMPROVED",
        "UNCHANGED",
        "DEGRADED",
    ]
}

assert sum(outcome_counts.values()) == 5

mean_inference_seconds = float(
    pilot_df09["inference_seconds"].mean()
)

# --------------------------------------------------
# 7. Hash frozen candidate source
# --------------------------------------------------

candidate_source_sha256 = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

# --------------------------------------------------
# 8. Software versions
# --------------------------------------------------

def package_version09(package):

    try:
        return metadata.version(package)

    except metadata.PackageNotFoundError:
        return "not_installed"


software_versions09 = {
    "python": sys.version.split()[0],
    "torch": package_version09("torch"),
    "transformers": package_version09("transformers"),
    "sentence_transformers": package_version09(
        "sentence-transformers"
    ),
    "numpy": package_version09("numpy"),
    "pandas": package_version09("pandas"),
}

# --------------------------------------------------
# 9. Build reproducibility manifest
# --------------------------------------------------

pilot_manifest09 = {

    "experiment": "cross_encoder_development_pilot_v1",

    "status": "pilot_only",

    "model": {
        "id": RERANK_MODEL_ID09,
        "revision": RERANK_REVISION09,
        "device": RERANK_DEVICE09,
        "max_length": RERANK_MAX_LENGTH09,
        "batch_size": RERANK_BATCH_SIZE09,
    },

    "dataset": {
        "partition": "dev",
        "query_type": "late",
        "n_queries": 5,
        "query_ids": expected_ids,
        "source_candidates": CANDIDATES_PATH09.name,
        "source_candidates_sha256": (
            candidate_source_sha256
        ),
    },

    "retrieval": {
        "baseline": "Hybrid RRF",
        "bm25_candidate_k": 50,
        "dense_candidate_k": 50,
        "rrf_k": 60,
        "rerank_top_k": 5,
    },

    "metrics": {
        "hybrid_recall_at_1": hybrid_r1,
        "reranker_recall_at_1": reranker_r1,
        "hybrid_recall_at_5": hybrid_r5,
        "reranker_recall_at_5": reranker_r5,
        "hybrid_mrr_at_5": hybrid_mrr5,
        "reranker_mrr_at_5": reranker_mrr5,
        "outcomes": outcome_counts,
    },

    "timing": {
        "total_pilot_seconds": float(total_elapsed09),
        "mean_inference_seconds_per_query": (
            mean_inference_seconds
        ),
        "model_loading_excluded": True,
        "device": "cpu",
    },

    "input_processing": {
        "article_input": "full article text",
        "max_total_pair_tokens": 512,
        "truncation": (
            "CrossEncoder tokenizer truncation"
        ),
    },

    "software_versions": software_versions09,

    "result_files": {
        "predictions": PILOT_PREDICTIONS_PATH09.name,
        "predictions_sha256": predictions_sha256,
        "metrics": PILOT_METRICS_PATH09.name,
        "metrics_sha256": metrics_sha256,
    },

    "limitations": [
        "Only five development queries were evaluated.",
        "The pilot subset is not representative of all queries.",
        "Queries are synthetic source-article excerpts.",
        "Only the initial Hybrid Top-5 candidates were reranked.",
        "Cross-Encoder input is limited to 512 total tokens.",
        "Long articles may be truncated.",
        "The pilot was executed on CPU.",
        "No evaluation-partition queries were used.",
        "The pilot is not a final benchmark.",
    ],
}

manifest_bytes = (
    json.dumps(
        pilot_manifest09,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
).encode("utf-8")

# --------------------------------------------------
# 10. Save without accidental overwrite
# --------------------------------------------------

files_to_save = [
    (PILOT_PREDICTIONS_PATH09, predictions_bytes),
    (PILOT_METRICS_PATH09, metrics_bytes),
    (PILOT_MANIFEST_PATH09, manifest_bytes),
]

for path, content in files_to_save:

    if path.exists():

        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )

        print("Existing file verified:", path.name)

    else:

        path.write_bytes(content)

        print("Saved:", path.name)

    assert path.read_bytes() == content

# --------------------------------------------------
# 11. Display final results
# --------------------------------------------------

print("\nPILOT METRICS")
print("-" * 55)

print(f"Hybrid Recall@1:      {hybrid_r1:.4f}")
print(f"Cross-Encoder R@1:    {reranker_r1:.4f}")

print(f"Hybrid Recall@5:      {hybrid_r5:.4f}")
print(f"Cross-Encoder R@5:    {reranker_r5:.4f}")

print(f"Hybrid MRR@5:         {hybrid_mrr5:.4f}")
print(f"Cross-Encoder MRR@5:   {reranker_mrr5:.4f}")

print("\nRANKING OUTCOMES")

for name, count in outcome_counts.items():
    print(f"{name}: {count}")

print("\nFILES")
print(PILOT_PREDICTIONS_PATH09)
print(PILOT_METRICS_PATH09)
print(PILOT_MANIFEST_PATH09)

print("\n" + "=" * 60)
print("CELL 16 VERIFICATION")
print("=" * 60)

print("Development queries:", len(pilot_df09))
print("Original candidate sets: VERIFIED")
print("Pilot metrics: SAVED")
print("Model revision: RECORDED")
print("Evaluation queries: NOT USED")
print("Original benchmark: UNCHANGED")

print("\nOK — Cross-Encoder pilot results saved")

In [ ]:
# CELL 17 — Development Audit & Reranking Feasibility
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU — No model inference

import json
import hashlib
import numpy as np
import pandas as pd

# ------------------------------------
# 1. Verify previous steps
# ------------------------------------

assert "dev_candidate_pools09" in globals()
assert "doc_by_id09" in globals()
assert "reranker09" in globals()
assert "pilot_df09" in globals()

assert len(dev_candidate_pools09) == 60
assert len(pilot_df09) == 5
assert RERANK_MAX_LENGTH09 == 512

# Verify saved pilot manifest
assert PILOT_MANIFEST_PATH09.is_file()

pilot_manifest17 = json.loads(
    PILOT_MANIFEST_PATH09.read_text(
        encoding="utf-8"
    )
)

assert pilot_manifest17["dataset"]["n_queries"] == 5

print("CELL 17 — DEVELOPMENT AUDIT")
print("=" * 60)

# ------------------------------------
# 2. Analyze all 60 development queries
# ------------------------------------

audit_rows17 = []

for q in dev_candidate_pools09:

    assert q["partition"] == "dev"
    assert q["query_type"] == "late"

    gold_id = q["source_id"]

    candidates = q["candidates"]

    gold_rank = next(
        (
            c["rank"]
            for c in candidates
            if c["document_id"] == gold_id
        ),
        None
    )

    assert gold_rank is not None

    # Tokenize the gold query/article pair.
    # This does not run inference.
    article_text = doc_by_id09[gold_id]["text"]

    encoded = reranker09.tokenizer(
        q["query"],
        article_text,
        truncation=False,
        add_special_tokens=True,
        verbose=False,
    )

    full_tokens = len(encoded["input_ids"])

    audit_rows17.append({
        "query_id": q["query_id"],
        "candidate_count": len(candidates),
        "gold_rank": gold_rank,
        "gold_in_top5": gold_rank <= 5,
        "gold_in_top10": gold_rank <= 10,
        "full_pair_tokens": full_tokens,
        "exceeds_512": full_tokens > 512,
    })

audit_df17 = pd.DataFrame(audit_rows17)

assert len(audit_df17) == 60
assert audit_df17["query_id"].is_unique

# ------------------------------------
# 3. Candidate coverage
# ------------------------------------

print("\nCANDIDATE COVERAGE — 60 LATE DEV QUERIES")
print("-" * 55)

recall1 = float(
    (audit_df17["gold_rank"] <= 1).mean()
)

recall5 = float(
    audit_df17["gold_in_top5"].mean()
)

recall10 = float(
    audit_df17["gold_in_top10"].mean()
)

print(f"Hybrid Recall@1:  {recall1:.4f}")
print(f"Hybrid Recall@5:  {recall5:.4f}")
print(f"Hybrid Recall@10: {recall10:.4f}")

print(
    "Gold outside Top-5:",
    int((~audit_df17["gold_in_top5"]).sum())
)

print(
    "Gold outside Top-10:",
    int((~audit_df17["gold_in_top10"]).sum())
)

# A Top-5 reranker cannot promote a document
# to rank 1 if it was outside the input Top-5.
print(
    "\nMaximum possible Recall@1 "
    "with Top-5 reranking:",
    f"{recall5:.4f}"
)

# ------------------------------------
# 4. Token truncation audit
# ------------------------------------

print("\nTOKEN LENGTH AUDIT — GOLD PAIRS")
print("-" * 55)

truncated_count = int(
    audit_df17["exceeds_512"].sum()
)

print("Pairs examined:", len(audit_df17))

print(
    "Pairs exceeding 512 tokens:",
    f"{truncated_count}/60"
)

print(
    "Truncation rate:",
    f"{audit_df17['exceeds_512'].mean():.4f}"
)

print(
    "Median full pair tokens:",
    round(
        audit_df17["full_pair_tokens"].median(),
        2
    )
)

print(
    "Maximum full pair tokens:",
    int(audit_df17["full_pair_tokens"].max())
)

# ------------------------------------
# 5. CPU runtime estimate
# ------------------------------------

mean_pilot_seconds = float(
    pilot_df09["inference_seconds"].mean()
)

dev_minutes = (
    60 * mean_pilot_seconds / 60
)

eval_late_minutes = (
    240 * mean_pilot_seconds / 60
)

print("\nESTIMATED CPU INFERENCE COST")
print("-" * 55)

print(
    "Measured pilot mean/query:",
    f"{mean_pilot_seconds:.2f} sec"
)

print(
    "60 development queries:",
    f"~{dev_minutes:.1f} minutes"
)

print(
    "240 late evaluation queries:",
    f"~{eval_late_minutes:.1f} minutes"
)

print(
    "Note: estimates assume Top-5 and similar "
    "input lengths; actual times may differ."
)

# ------------------------------------
# 6. Development audit preview
# ------------------------------------

print("\nDEVELOPMENT AUDIT PREVIEW")

display(
    audit_df17.head(10)
)

# ------------------------------------
# 7. Verify no evaluation contamination
# ------------------------------------

assert all(
    q["partition"] == "dev"
    for q in dev_candidate_pools09
)

assert np.isclose(recall1, 0.60)
assert np.isclose(recall5, 0.85)
assert np.isclose(recall10, 59 / 60)

print("\n" + "=" * 60)
print("CELL 17 VERIFICATION")
print("=" * 60)

print("Development queries audited:", len(audit_df17))
print("Cross-Encoder inference: NOT EXECUTED")
print("Candidate coverage: VERIFIED")
print("Input lengths: AUDITED")
print("Evaluation queries: NOT USED")
print("Existing files: UNCHANGED")

print("\nOK — Development feasibility audit completed")

In [ ]:
# CELL 18 — Freeze Cross-Encoder Evaluation Protocol
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU

import json
import hashlib
import numpy as np
from pathlib import Path

# ------------------------------------
# 1. Verify completed experiments
# ------------------------------------

assert len(dev_candidate_pools09) == 60
assert len(audit_df17) == 60
assert len(pilot_df09) == 5

assert PILOT_MANIFEST_PATH09.is_file()
assert CANDIDATES_PATH09.is_file()

assert RERANK_MODEL_ID09 == "BAAI/bge-reranker-v2-m3"
assert RERANK_MAX_LENGTH09 == 512
assert RERANK_DEVICE09 == "cpu"

print("CELL 18 — FREEZE RERANKING PROTOCOL")
print("=" * 60)

# ------------------------------------
# 2. Final experiment configuration
# ------------------------------------

RERANK_TOP_K09 = 5

DEV_QUERY_COUNT09 = 60
EVAL_LATE_QUERY_COUNT09 = 240

assert RERANK_TOP_K09 == PILOT_TOP_K09

assert CANDIDATE_K09 == 50
assert RRF_K09 == 60

print("\nEXPERIMENT CONFIGURATION")
print("-" * 55)

print("Model:", RERANK_MODEL_ID09)
print("Model revision:", RERANK_REVISION09)
print("Device:", RERANK_DEVICE09)
print("Cross-Encoder max tokens:", RERANK_MAX_LENGTH09)
print("Rerank Top-K:", RERANK_TOP_K09)
print("BM25 candidate K:", CANDIDATE_K09)
print("Dense candidate K:", CANDIDATE_K09)
print("RRF k:", RRF_K09)

# ------------------------------------
# 3. Verify development candidate source
# ------------------------------------

candidate_sha18 = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

assert candidate_sha18 == (
    "f95253f6fa83388e28880f40e21c2eac379d9b394ae412510ab1f199d7ccb057"
)

pilot_manifest18 = json.loads(
    PILOT_MANIFEST_PATH09.read_text(
        encoding="utf-8"
    )
)

assert pilot_manifest18["model"]["revision"] == (
    RERANK_REVISION09
)

assert pilot_manifest18["retrieval"]["rerank_top_k"] == 5

assert pilot_manifest18["dataset"][
    "source_candidates_sha256"
] == candidate_sha18

# ------------------------------------
# 4. Calculate development constraints
# ------------------------------------

baseline_r1 = float(
    (audit_df17["gold_rank"] == 1).mean()
)

baseline_r5 = float(
    (audit_df17["gold_rank"] <= 5).mean()
)

baseline_r10 = float(
    (audit_df17["gold_rank"] <= 10).mean()
)

truncation_rate = float(
    audit_df17["exceeds_512"].mean()
)

assert np.isclose(baseline_r1, 0.60)
assert np.isclose(baseline_r5, 0.85)
assert np.isclose(baseline_r10, 59 / 60)

# ------------------------------------
# 5. Construct protocol
# ------------------------------------

rerank_protocol18 = {

    "experiment": "cross_encoder_reranking_v1",

    "status": "protocol_frozen",

    "model": {
        "id": RERANK_MODEL_ID09,
        "revision": RERANK_REVISION09,
        "device": "cpu",
        "max_length": 512,
        "batch_size": RERANK_BATCH_SIZE09,
    },

    "retrieval": {
        "baseline": "Hybrid RRF",
        "bm25_candidate_k": 50,
        "dense_candidate_k": 50,
        "rrf_k": 60,
        "rerank_top_k": 5,
        "candidate_order": "Hybrid RRF rank",
        "rerank_sort": (
            "Descending Cross-Encoder score; "
            "tie-break by Hybrid rank and document ID"
        ),
        "remaining_candidates": (
            "Retain original Hybrid RRF ordering "
            "after the reranked Top-5"
        ),
    },

    "development": {
        "partition": "dev",
        "query_type": "late",
        "n_queries": DEV_QUERY_COUNT09,
        "candidate_file": CANDIDATES_PATH09.name,
        "candidate_sha256": candidate_sha18,
        "baseline_recall_at_1": baseline_r1,
        "baseline_recall_at_5": baseline_r5,
        "baseline_recall_at_10": baseline_r10,
        "maximum_possible_recall_at_1": baseline_r5,
    },

    "evaluation_plan": {
        "late_queries": EVAL_LATE_QUERY_COUNT09,
        "lead_queries": 240,
        "use_for_hyperparameter_selection": False,
        "execute_after_development": True,
    },

    "metrics": [
        "recall@1",
        "recall@5",
        "recall@10",
        "mrr@10",
    ],

    "execution": {
        "checkpointing": True,
        "reuse_verified_pilot_results": True,
        "model_training": False,
    },

    "limitations": [
        "Synthetic source-article queries.",
        "Single relevant source document per query.",
        "Full-text BM25 has an exact-match advantage.",
        "Top-5 reranking has an 85% development "
        "Recall@1 upper bound.",
        "Cross-Encoder truncates pairs to 512 tokens.",
        "59 of 60 development gold pairs exceed "
        "the 512-token limit.",
        "CPU inference is comparatively expensive.",
        "No evaluation queries used for model selection.",
        "Pilot results do not establish generalization.",
    ],
}

# ------------------------------------
# 6. Save without accidental overwrite
# ------------------------------------

PROTOCOL_PATH18 = (
    RERANK_DIR09 / "reranking_protocol_v1.json"
)

protocol_bytes18 = (
    json.dumps(
        rerank_protocol18,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

if PROTOCOL_PATH18.exists():

    assert (
        PROTOCOL_PATH18.read_bytes()
        == protocol_bytes18
    ), "Existing protocol differs."

    print("\nExisting protocol verified.")

else:

    PROTOCOL_PATH18.write_bytes(
        protocol_bytes18
    )

    print("\nProtocol saved.")

assert (
    PROTOCOL_PATH18.read_bytes()
    == protocol_bytes18
)

protocol_sha18 = hashlib.sha256(
    protocol_bytes18
).hexdigest()

# ------------------------------------
# 7. Final verification
# ------------------------------------

print("\n" + "=" * 60)
print("CELL 18 VERIFICATION")
print("=" * 60)

print("Model: VERIFIED")
print("Top-K strategy: FROZEN")
print("Development queries:", DEV_QUERY_COUNT09)
print("Evaluation queries used: 0")
print("Candidate SHA256: VERIFIED")
print("Pilot configuration: MATCHED")
print("Protocol SHA256:", protocol_sha18)

print("\nSaved to:")
print(PROTOCOL_PATH18)

print("\nOK — Reranking protocol frozen")

In [ ]:
# CELL 19 — Checkpointed Cross-Encoder Development Benchmark
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU
# Queries: late_006 to late_010

import json
import time
import hashlib
import re
import numpy as np
import pandas as pd

from pathlib import Path

# --------------------------------------------------
# 1. Verify frozen experiment
# --------------------------------------------------

assert "reranker09" in globals()
assert "dev_candidate_pools09" in globals()
assert "doc_by_id09" in globals()

assert len(dev_candidate_pools09) == 60

assert RERANK_TOP_K09 == 5
assert RERANK_MAX_LENGTH09 == 512
assert RERANK_DEVICE09 == "cpu"

assert PROTOCOL_PATH18.is_file()
assert CANDIDATES_PATH09.is_file()
assert PILOT_PREDICTIONS_PATH09.is_file()
assert PILOT_MANIFEST_PATH09.is_file()

protocol_bytes19 = PROTOCOL_PATH18.read_bytes()

protocol_sha19 = hashlib.sha256(
    protocol_bytes19
).hexdigest()

protocol19 = json.loads(protocol_bytes19)

candidate_sha19 = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

assert protocol19["model"]["revision"] == (
    RERANK_REVISION09
)

assert protocol19["development"]["candidate_sha256"] == (
    candidate_sha19
)

assert protocol19["retrieval"]["rerank_top_k"] == 5

print("FROZEN PROTOCOL VERIFIED")
print("=" * 60)

print("Model:", RERANK_MODEL_ID09)
print("Device:", RERANK_DEVICE09)
print("Rerank Top-K:", RERANK_TOP_K09)
print("Protocol SHA256:", protocol_sha19)

# --------------------------------------------------
# 2. Verify saved pilot
# --------------------------------------------------

pilot_manifest19 = json.loads(
    PILOT_MANIFEST_PATH09.read_text(
        encoding="utf-8"
    )
)

assert hashlib.sha256(
    PILOT_PREDICTIONS_PATH09.read_bytes()
).hexdigest() == (
    pilot_manifest19["result_files"][
        "predictions_sha256"
    ]
)

with PILOT_PREDICTIONS_PATH09.open(
    encoding="utf-8"
) as f:

    saved_pilot19 = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(saved_pilot19) == 5

print("\nSAVED PILOT VERIFIED")
print("Previously completed queries:", len(saved_pilot19))
print("Pilot inference will NOT be repeated.")

# --------------------------------------------------
# 3. Checkpoint directory
# --------------------------------------------------

CHECKPOINT_DIR19 = (
    RERANK_DIR09 / "dev_top5_checkpoints"
)

CHECKPOINT_DIR19.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# 4. Reusable batch function
# --------------------------------------------------

def run_dev_checkpoint_batch09(start_index, end_index):

    assert 0 <= start_index < end_index <= 60

    batch_queries = dev_candidate_pools09[
        start_index:end_index
    ]

    records = []

    print("\nCROSS-ENCODER DEVELOPMENT BATCH")
    print("=" * 60)

    print("Start index:", start_index)
    print("End index:", end_index)
    print("Queries:", len(batch_queries))

    for index, query_record in enumerate(
        batch_queries,
        start=1
    ):

        query_id = query_record["query_id"]
        gold_id = query_record["source_id"]

        assert re.fullmatch(
            r"late_\d{3}",
            query_id
        )

        checkpoint_path = (
            CHECKPOINT_DIR19 / f"{query_id}.json"
        )

        # Original Hybrid ordering
        all_candidates = query_record["candidates"]

        all_hybrid_ids = [
            c["document_id"]
            for c in all_candidates
        ]

        top5 = all_candidates[:RERANK_TOP_K09]

        assert len(top5) == 5

        original_top5_ids = [
            c["document_id"]
            for c in top5
        ]

        assert len(set(original_top5_ids)) == 5

        hybrid_gold_rank = (
            all_hybrid_ids.index(gold_id) + 1
            if gold_id in all_hybrid_ids
            else None
        )

        # ------------------------------------------
        # A. Reuse checkpoint if already completed
        # ------------------------------------------

        if checkpoint_path.exists():

            record = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            assert record["query_id"] == query_id

            assert record["source_id"] == gold_id

            assert record["protocol_sha256"] == (
                protocol_sha19
            )

            assert record["candidate_sha256"] == (
                candidate_sha19
            )

            assert record["model_revision"] == (
                RERANK_REVISION09
            )

            assert record["hybrid_top5_ids"] == (
                original_top5_ids
            )

            assert record["hybrid_gold_rank"] == (
                hybrid_gold_rank
            )

            print(
                f"[{index}/{len(batch_queries)}] "
                f"{query_id}: CHECKPOINT VERIFIED"
            )

            records.append(record)
            continue

        # ------------------------------------------
        # B. Prepare the five input pairs
        # ------------------------------------------

        pairs = [
            (
                query_record["query"],
                doc_by_id09[c["document_id"]]["text"]
            )
            for c in top5
        ]

        print(
            f"[{index}/{len(batch_queries)}] "
            f"Running {query_id}...",
            flush=True
        )

        # ------------------------------------------
        # C. Cross-Encoder inference
        # ------------------------------------------

        start_time = time.perf_counter()

        scores = reranker09.predict(
            pairs,
            batch_size=RERANK_BATCH_SIZE09,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

        elapsed = (
            time.perf_counter() - start_time
        )

        scores = np.asarray(
            scores,
            dtype=float
        ).reshape(-1)

        assert scores.shape == (5,)
        assert np.isfinite(scores).all()

        # ------------------------------------------
        # D. Deterministic reranking
        # ------------------------------------------

        reranked_top5 = [
            {
                "document_id": candidate["document_id"],
                "hybrid_rank": int(candidate["rank"]),
                "reranker_score": float(score),
            }
            for candidate, score in zip(
                top5,
                scores
            )
        ]

        reranked_top5.sort(
            key=lambda row: (
                -row["reranker_score"],
                row["hybrid_rank"],
                row["document_id"],
            )
        )

        reranked_top5_ids = [
            row["document_id"]
            for row in reranked_top5
        ]

        assert set(reranked_top5_ids) == (
            set(original_top5_ids)
        )

        # Preserve the original Hybrid ordering
        # of all candidates outside Top-5.
        full_reranked_ids = (
            reranked_top5_ids
            + all_hybrid_ids[5:]
        )

        reranker_gold_rank = (
            full_reranked_ids.index(gold_id) + 1
            if gold_id in full_reranked_ids
            else None
        )

        # ------------------------------------------
        # E. Compare ranking
        # ------------------------------------------

        hybrid_rr = (
            1.0 / hybrid_gold_rank
            if hybrid_gold_rank is not None
            else 0.0
        )

        reranker_rr = (
            1.0 / reranker_gold_rank
            if reranker_gold_rank is not None
            else 0.0
        )

        if reranker_rr > hybrid_rr:
            outcome = "IMPROVED"
        elif reranker_rr < hybrid_rr:
            outcome = "DEGRADED"
        else:
            outcome = "UNCHANGED"

        # ------------------------------------------
        # F. Build checkpoint record
        # ------------------------------------------

        record = {
            "query_id": query_id,
            "source_id": gold_id,
            "partition": "dev",
            "query_type": "late",
            "model_id": RERANK_MODEL_ID09,
            "model_revision": RERANK_REVISION09,
            "protocol_sha256": protocol_sha19,
            "candidate_sha256": candidate_sha19,
            "rerank_top_k": 5,
            "hybrid_top5_ids": original_top5_ids,
            "reranked_top5": reranked_top5,
            "hybrid_gold_rank": hybrid_gold_rank,
            "reranker_gold_rank": reranker_gold_rank,
            "outcome": outcome,
            "inference_seconds": float(elapsed),
        }

        checkpoint_bytes = (
            json.dumps(
                record,
                ensure_ascii=False,
                indent=2,
                sort_keys=True,
                allow_nan=False,
            ) + "\n"
        ).encode("utf-8")

        # ------------------------------------------
        # G. Save immediately after each query
        # ------------------------------------------

        assert not checkpoint_path.exists()

        checkpoint_path.write_bytes(
            checkpoint_bytes
        )

        assert checkpoint_path.read_bytes() == (
            checkpoint_bytes
        )

        print(
            f"  Hybrid rank={hybrid_gold_rank}"
            f" | Reranker rank={reranker_gold_rank}"
            f" | {outcome}"
        )

        print(
            f"  Inference: {elapsed:.2f} sec"
            " | SAVED",
            flush=True
        )

        records.append(record)

    return records


# --------------------------------------------------
# 5. Run only queries 6–10
# --------------------------------------------------

dev_batch19 = run_dev_checkpoint_batch09(
    start_index=5,
    end_index=10,
)

assert len(dev_batch19) == 5

# --------------------------------------------------
# 6. Summarize this batch
# --------------------------------------------------

batch_df19 = pd.DataFrame([
    {
        "query_id": r["query_id"],
        "hybrid_gold_rank": r["hybrid_gold_rank"],
        "reranker_gold_rank": r["reranker_gold_rank"],
        "outcome": r["outcome"],
        "inference_seconds": r["inference_seconds"],
    }
    for r in dev_batch19
])

print("\nBATCH RESULTS")
print("-" * 60)

display(batch_df19.round(3))

print("\nRANKING OUTCOMES")

print(
    batch_df19["outcome"]
    .value_counts()
    .to_string()
)

print("\nMean inference time per query:")

print(
    f"{batch_df19['inference_seconds'].mean():.2f} sec"
)

# --------------------------------------------------
# 7. Final verification
# --------------------------------------------------

assert all(
    (
        CHECKPOINT_DIR19 / f"{q['query_id']}.json"
    ).is_file()
    for q in dev_candidate_pools09[5:10]
)

print("\n" + "=" * 60)
print("CELL 19 VERIFICATION")
print("=" * 60)

print("Previous pilot queries:", 5)
print("New batch queries:", len(dev_batch19))
print("Development queries processed:", 10)
print("Individual checkpoints: VERIFIED")
print("Evaluation queries: NOT USED")

print("\nCheckpoint directory:")
print(CHECKPOINT_DIR19)

print("\nOK — Development batch checkpointed")

In [ ]:
# CELL 20 — Cross-Encoder Development Batch 11–15
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU

import pandas as pd
import json
import hashlib

# ------------------------------------
# 1. Verify previous steps
# ------------------------------------

assert "run_dev_checkpoint_batch09" in globals(), (
    "Run Cell 19 first to define the batch function."
)

assert "dev_candidate_pools09" in globals()
assert "CHECKPOINT_DIR19" in globals()

assert len(dev_candidate_pools09) == 60
assert RERANK_TOP_K09 == 5

assert PROTOCOL_PATH18.is_file()
assert CANDIDATES_PATH09.is_file()

print("CELL 20 — DEVELOPMENT BATCH 11–15")
print("=" * 60)

# ------------------------------------
# 2. Verify frozen protocol
# ------------------------------------

current_protocol_sha20 = hashlib.sha256(
    PROTOCOL_PATH18.read_bytes()
).hexdigest()

assert current_protocol_sha20 == protocol_sha19

print("Frozen protocol: VERIFIED")

# ------------------------------------
# 3. Select next five queries
# ------------------------------------

batch_start20 = 10
batch_end20 = 15

expected_ids20 = [
    q["query_id"]
    for q in dev_candidate_pools09[
        batch_start20:batch_end20
    ]
]

assert len(expected_ids20) == 5

print("\nQUERIES TO PROCESS")
print("-" * 50)

for query_id in expected_ids20:
    print(query_id)

# ------------------------------------
# 4. Run checkpointed inference
# ------------------------------------

batch_records20 = run_dev_checkpoint_batch09(
    start_index=batch_start20,
    end_index=batch_end20,
)

assert len(batch_records20) == 5

assert [
    record["query_id"]
    for record in batch_records20
] == expected_ids20

# ------------------------------------
# 5. Verify saved checkpoints
# ------------------------------------

for record in batch_records20:

    path = (
        CHECKPOINT_DIR19
        / f"{record['query_id']}.json"
    )

    assert path.is_file()

    saved = json.loads(
        path.read_text(encoding="utf-8")
    )

    assert saved == record

    assert saved["protocol_sha256"] == (
        current_protocol_sha20
    )

print("\nAll 5 checkpoints verified.")

# ------------------------------------
# 6. Build batch comparison table
# ------------------------------------

batch_df20 = pd.DataFrame([
    {
        "query_id": record["query_id"],
        "hybrid_gold_rank": record[
            "hybrid_gold_rank"
        ],
        "reranker_gold_rank": record[
            "reranker_gold_rank"
        ],
        "outcome": record["outcome"],
        "inference_seconds": record[
            "inference_seconds"
        ],
    }
    for record in batch_records20
])

print("\nBATCH RESULTS")
print("-" * 55)

display(batch_df20.round(3))

# ------------------------------------
# 7. Calculate batch metrics
# ------------------------------------

hybrid_r1_20 = (
    batch_df20["hybrid_gold_rank"] == 1
).mean()

reranker_r1_20 = (
    batch_df20["reranker_gold_rank"] == 1
).mean()

print("\nBATCH METRICS")
print("-" * 55)

print(
    f"Hybrid Recall@1: {hybrid_r1_20:.4f}"
)

print(
    f"Cross-Encoder Recall@1: {reranker_r1_20:.4f}"
)

print("\nRANKING OUTCOMES")

for outcome in [
    "IMPROVED",
    "UNCHANGED",
    "DEGRADED",
]:
    count = int(
        (batch_df20["outcome"] == outcome).sum()
    )

    print(f"{outcome}: {count}")

# ------------------------------------
# 8. Runtime statistics
# ------------------------------------

print("\nPERFORMANCE")
print("-" * 55)

print(
    "Mean inference per query:",
    f"{batch_df20['inference_seconds'].mean():.2f} sec"
)

print(
    "Total inference time:",
    f"{batch_df20['inference_seconds'].sum():.2f} sec"
)

# ------------------------------------
# 9. Final verification
# ------------------------------------

assert len(batch_df20) == 5
assert batch_df20["query_id"].is_unique

print("\n" + "=" * 60)
print("CELL 20 VERIFICATION")
print("=" * 60)

print("Pilot queries:", 5)
print("Cell 19 queries:", 5)
print("Cell 20 queries:", 5)

print("Total processed development queries: 15/60")
print("New checkpoints: VERIFIED")
print("Evaluation queries: NOT USED")
print("Original benchmark: UNCHANGED")

print("\nOK — Development batch 11–15 completed")

In [ ]:
# CELL 21 — Consolidated Development Evaluation (15/60)
# CPU only — No model inference

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

RESULTS_DIR21 = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/reranking_eval_v1"
)

PROTOCOL21 = RESULTS_DIR21 / "reranking_protocol_v1.json"
CANDIDATES21 = RESULTS_DIR21 / "dev_late_candidates.jsonl"

PILOT_PRED21 = RESULTS_DIR21 / "pilot_top5_predictions.jsonl"
PILOT_METRICS21 = RESULTS_DIR21 / "pilot_top5_metrics.csv"
PILOT_MANIFEST21 = RESULTS_DIR21 / "pilot_top5_manifest.json"

CHECKPOINTS21 = RESULTS_DIR21 / "dev_top5_checkpoints"

assert RESULTS_DIR21.is_dir()

print("CELL 21 — CONSOLIDATED DEVELOPMENT EVALUATION")
print("=" * 60)

# --------------------------------------------------
# 2. File verification
# --------------------------------------------------

def sha256_21(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

protocol21 = json.loads(
    PROTOCOL21.read_text(encoding="utf-8")
)

pilot_manifest21 = json.loads(
    PILOT_MANIFEST21.read_text(encoding="utf-8")
)

protocol_sha21 = sha256_21(PROTOCOL21)
candidate_sha21 = sha256_21(CANDIDATES21)

assert (
    candidate_sha21
    == protocol21["development"]["candidate_sha256"]
)

assert (
    sha256_21(PILOT_PRED21)
    == pilot_manifest21["result_files"]["predictions_sha256"]
)

assert (
    sha256_21(PILOT_METRICS21)
    == pilot_manifest21["result_files"]["metrics_sha256"]
)

assert (
    pilot_manifest21["model"]["revision"]
    == protocol21["model"]["revision"]
)

assert (
    pilot_manifest21["dataset"]["source_candidates_sha256"]
    == candidate_sha21
)

assert protocol21["retrieval"]["rerank_top_k"] == 5

print("Frozen protocol: VERIFIED")
print("Candidate dataset: VERIFIED")
print("Pilot results: VERIFIED")

# --------------------------------------------------
# 3. Load saved data
# --------------------------------------------------

def read_jsonl21(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

candidate_pools21 = read_jsonl21(CANDIDATES21)
pilot_predictions21 = read_jsonl21(PILOT_PRED21)
pilot_metrics_df21 = pd.read_csv(PILOT_METRICS21)

assert len(candidate_pools21) == 60
assert len(pilot_predictions21) == 5
assert len(pilot_metrics_df21) == 5

pilot_by_id21 = {
    row["query_id"]: row
    for row in pilot_predictions21
}

pilot_metrics_by_id21 = pilot_metrics_df21.set_index(
    "query_id"
)

# --------------------------------------------------
# 4. Load and verify all 15 results
# --------------------------------------------------

combined_rows21 = []

for index, pool in enumerate(candidate_pools21[:15]):

    query_id = pool["query_id"]
    gold_id = pool["source_id"]

    hybrid_ids = [
        c["document_id"]
        for c in pool["candidates"]
    ]

    hybrid_top5 = hybrid_ids[:5]

    assert len(hybrid_top5) == 5
    assert gold_id in hybrid_ids

    # First 5: saved pilot
    if index < 5:

        saved = pilot_by_id21[query_id]

        assert saved["source_id"] == gold_id
        assert saved["hybrid_ids"] == hybrid_top5

        reranked = saved["reranked_candidates"]

        inference_seconds = float(
            pilot_metrics_by_id21.loc[
                query_id, "inference_seconds"
            ]
        )

        source = "pilot"

    # Next 10: individual checkpoints
    else:

        checkpoint_path = (
            CHECKPOINTS21 / f"{query_id}.json"
        )

        assert checkpoint_path.is_file(), checkpoint_path

        saved = json.loads(
            checkpoint_path.read_text(
                encoding="utf-8"
            )
        )

        assert saved["query_id"] == query_id
        assert saved["source_id"] == gold_id

        assert saved["protocol_sha256"] == protocol_sha21
        assert saved["candidate_sha256"] == candidate_sha21

        assert saved["model_revision"] == (
            protocol21["model"]["revision"]
        )

        assert saved["hybrid_top5_ids"] == hybrid_top5

        reranked = saved["reranked_top5"]

        inference_seconds = float(
            saved["inference_seconds"]
        )

        source = "checkpoint"

    # --------------------------------------------------
    # 5. Verify reranked candidates
    # --------------------------------------------------

    reranked_ids = [
        row["document_id"]
        for row in reranked
    ]

    assert len(reranked_ids) == 5
    assert set(reranked_ids) == set(hybrid_top5)

    assert all(
        np.isfinite(row["reranker_score"])
        for row in reranked
    )

    # Verify deterministic sorting.
    assert reranked == sorted(
        reranked,
        key=lambda row: (
            -row["reranker_score"],
            row["hybrid_rank"],
            row["document_id"],
        )
    )

    # Preserve original Hybrid positions after Top-5.
    full_reranked_ids = (
        reranked_ids + hybrid_ids[5:]
    )

    hybrid_rank = hybrid_ids.index(gold_id) + 1

    reranker_rank = (
        full_reranked_ids.index(gold_id) + 1
    )

    if reranker_rank < hybrid_rank:
        outcome = "IMPROVED"
    elif reranker_rank > hybrid_rank:
        outcome = "DEGRADED"
    else:
        outcome = "UNCHANGED"

    # Verify checkpoint's recorded ranking.
    if source == "checkpoint":

        assert saved["hybrid_gold_rank"] == hybrid_rank
        assert saved["reranker_gold_rank"] == reranker_rank
        assert saved["outcome"] == outcome

    else:

        pilot_row = pilot_metrics_by_id21.loc[query_id]

        assert int(pilot_row["hybrid_gold_rank"]) == hybrid_rank
        assert int(pilot_row["reranker_gold_rank"]) == reranker_rank
        assert pilot_row["outcome"] == outcome

    combined_rows21.append({
        "query_id": query_id,
        "hybrid_rank": hybrid_rank,
        "reranker_rank": reranker_rank,
        "hybrid_rr10": (
            1.0 / hybrid_rank if hybrid_rank <= 10 else 0.0
        ),
        "reranker_rr10": (
            1.0 / reranker_rank if reranker_rank <= 10 else 0.0
        ),
        "outcome": outcome,
        "inference_seconds": inference_seconds,
        "source": source,
    })

# --------------------------------------------------
# 6. Build consolidated dataframe
# --------------------------------------------------

combined_df21 = pd.DataFrame(combined_rows21)

assert len(combined_df21) == 15
assert combined_df21["query_id"].is_unique

print("\nQUERY-LEVEL RESULTS")
print("-" * 60)

display(combined_df21.round(4))

# --------------------------------------------------
# 7. Aggregate metrics
# --------------------------------------------------

def ranking_metrics21(ranks):

    ranks = np.asarray(ranks, dtype=float)

    return {
        "recall@1": float(np.mean(ranks <= 1)),
        "recall@5": float(np.mean(ranks <= 5)),
        "recall@10": float(np.mean(ranks <= 10)),
        "mrr@10": float(
            np.mean(
                np.where(ranks <= 10, 1.0 / ranks, 0.0)
            )
        ),
    }

hybrid_metrics21 = ranking_metrics21(
    combined_df21["hybrid_rank"]
)

reranker_metrics21 = ranking_metrics21(
    combined_df21["reranker_rank"]
)

metrics_table21 = pd.DataFrame({
    "Hybrid RRF": hybrid_metrics21,
    "Cross-Encoder": reranker_metrics21,
})

metrics_table21["Difference"] = (
    metrics_table21["Cross-Encoder"]
    - metrics_table21["Hybrid RRF"]
)

print("\nCONSOLIDATED DEVELOPMENT METRICS")
print("=" * 60)

display(metrics_table21.round(4))

# --------------------------------------------------
# 8. Ranking outcomes and runtime
# --------------------------------------------------

print("\nRANKING OUTCOMES")
print("-" * 55)

for name in ["IMPROVED", "UNCHANGED", "DEGRADED"]:

    count = int(
        (combined_df21["outcome"] == name).sum()
    )

    print(f"{name}: {count}")

print("\nCPU INFERENCE")

print(
    "Queries completed:",
    len(combined_df21),
    "/ 60"
)

print(
    "Total measured inference:",
    f"{combined_df21['inference_seconds'].sum():.2f} sec"
)

print(
    "Mean per query:",
    f"{combined_df21['inference_seconds'].mean():.2f} sec"
)

# --------------------------------------------------
# 9. Save consolidated progress
# --------------------------------------------------

SUMMARY_PATH21 = (
    RESULTS_DIR21 / "dev_top5_progress_15.csv"
)

summary_bytes21 = combined_df21.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.8f",
).encode("utf-8")

if SUMMARY_PATH21.exists():

    assert SUMMARY_PATH21.read_bytes() == summary_bytes21

    print("\nExisting summary verified.")

else:

    SUMMARY_PATH21.write_bytes(summary_bytes21)

    print("\nSummary saved.")

assert SUMMARY_PATH21.read_bytes() == summary_bytes21

# --------------------------------------------------
# 10. Final verification
# --------------------------------------------------

print("\n" + "=" * 60)
print("CELL 21 VERIFICATION")
print("=" * 60)

print("Development queries:", len(combined_df21))
print("Saved pilot queries:", 5)
print("Verified checkpoints:", 10)
print("Cross-Encoder inference: NOT EXECUTED")
print("Evaluation queries: NOT USED")
print("Original benchmark files: UNCHANGED")

print("\nSaved to:")
print(SUMMARY_PATH21)

print("\nOK — Consolidated development evaluation completed")

In [ ]:
# CELL 22 — Cross-Encoder Development Batch 16–20
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU

import json
import hashlib
import numpy as np
import pandas as pd

# ------------------------------------
# 1. Verify previous steps
# ------------------------------------

required22 = [
    "run_dev_checkpoint_batch09",
    "dev_candidate_pools09",
    "CHECKPOINT_DIR19",
    "PROTOCOL_PATH18",
    "CANDIDATES_PATH09",
]

missing22 = [
    name for name in required22
    if name not in globals()
]

assert not missing22, (
    "Missing variables: " + ", ".join(missing22)
)

assert len(dev_candidate_pools09) == 60
assert RERANK_TOP_K09 == 5

print("CELL 22 — DEVELOPMENT BATCH 16–20")
print("=" * 60)

# ------------------------------------
# 2. Verify frozen protocol
# ------------------------------------

protocol_sha22 = hashlib.sha256(
    PROTOCOL_PATH18.read_bytes()
).hexdigest()

candidate_sha22 = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

protocol22 = json.loads(
    PROTOCOL_PATH18.read_text(encoding="utf-8")
)

assert protocol_sha22 == protocol_sha19

assert candidate_sha22 == (
    protocol22["development"]["candidate_sha256"]
)

assert protocol22["retrieval"]["rerank_top_k"] == 5

assert protocol22["model"]["revision"] == (
    RERANK_REVISION09
)

print("Frozen protocol: VERIFIED")
print("Candidate dataset: VERIFIED")

# ------------------------------------
# 3. Verify previous results
# ------------------------------------

assert PILOT_PREDICTIONS_PATH09.is_file()

assert (
    RESULTS_DIR21 / "dev_top5_progress_15.csv"
).is_file()

for q in dev_candidate_pools09[5:15]:

    checkpoint = (
        CHECKPOINT_DIR19 / f"{q['query_id']}.json"
    )

    assert checkpoint.is_file(), checkpoint

print("Previous 15 queries: SAVED")

# ------------------------------------
# 4. Select queries 16–20
# ------------------------------------

START22 = 15
END22 = 20

selected22 = dev_candidate_pools09[
    START22:END22
]

expected_ids22 = [
    q["query_id"] for q in selected22
]

assert expected_ids22 == [
    f"late_{i:03d}"
    for i in range(16, 21)
]

print("\nQUERIES")
print("-" * 50)

for query_id in expected_ids22:
    print(query_id)

# ------------------------------------
# 5. Run checkpointed inference
# ------------------------------------

batch_records22 = run_dev_checkpoint_batch09(
    start_index=START22,
    end_index=END22,
)

assert len(batch_records22) == 5

assert [
    r["query_id"] for r in batch_records22
] == expected_ids22

# ------------------------------------
# 6. Verify checkpoints
# ------------------------------------

for record in batch_records22:

    path = (
        CHECKPOINT_DIR19
        / f"{record['query_id']}.json"
    )

    assert path.is_file()

    saved = json.loads(
        path.read_text(encoding="utf-8")
    )

    assert saved == record

    assert saved["protocol_sha256"] == protocol_sha22
    assert saved["candidate_sha256"] == candidate_sha22
    assert saved["model_revision"] == RERANK_REVISION09

print("\nAll 5 checkpoints verified.")

# ------------------------------------
# 7. Build batch results
# ------------------------------------

batch_df22 = pd.DataFrame([
    {
        "query_id": r["query_id"],
        "hybrid_rank": r["hybrid_gold_rank"],
        "reranker_rank": r["reranker_gold_rank"],
        "outcome": r["outcome"],
        "inference_seconds": r["inference_seconds"],
    }
    for r in batch_records22
])

print("\nQUERY-LEVEL RESULTS")
print("-" * 55)

display(batch_df22.round(3))

# ------------------------------------
# 8. Calculate batch metrics
# ------------------------------------

def recall_at_1_22(column):

    return float((column == 1).mean())


def mrr_at_10_22(column):

    ranks = column.to_numpy()

    reciprocal_ranks = [
        1.0 / rank
        if pd.notna(rank) and 1 <= rank <= 10
        else 0.0
        for rank in ranks
    ]

    return float(np.mean(reciprocal_ranks))


hybrid_r1_22 = recall_at_1_22(
    batch_df22["hybrid_rank"]
)

reranker_r1_22 = recall_at_1_22(
    batch_df22["reranker_rank"]
)

hybrid_mrr_22 = mrr_at_10_22(
    batch_df22["hybrid_rank"]
)

reranker_mrr_22 = mrr_at_10_22(
    batch_df22["reranker_rank"]
)

print("\nBATCH METRICS")
print("=" * 55)

print(f"Hybrid Recall@1:       {hybrid_r1_22:.4f}")
print(f"Cross-Encoder Recall@1: {reranker_r1_22:.4f}")

print(f"Hybrid MRR@10:          {hybrid_mrr_22:.4f}")
print(f"Cross-Encoder MRR@10:    {reranker_mrr_22:.4f}")

print("\nRANKING OUTCOMES")

for outcome in [
    "IMPROVED",
    "UNCHANGED",
    "DEGRADED",
]:

    count = int(
        (batch_df22["outcome"] == outcome).sum()
    )

    print(f"{outcome}: {count}")

# ------------------------------------
# 9. Runtime statistics
# ------------------------------------

print("\nCPU PERFORMANCE")
print("-" * 55)

print(
    "Mean inference per query:",
    f"{batch_df22['inference_seconds'].mean():.2f} sec"
)

print(
    "Total inference:",
    f"{batch_df22['inference_seconds'].sum():.2f} sec"
)

# ------------------------------------
# 10. Final verification
# ------------------------------------

assert len(batch_df22) == 5
assert batch_df22["query_id"].is_unique

print("\n" + "=" * 60)
print("CELL 22 VERIFICATION")
print("=" * 60)

print("Development queries completed: 20/60")
print("Current batch: late_016 to late_020")
print("Checkpoints: VERIFIED")
print("Evaluation queries: NOT USED")
print("Original benchmark: UNCHANGED")

print("\nOK — Development batch 16–20 completed")

In [ ]:
# CELL 23 — Consolidated Development Evaluation (20/60)
# CPU only — No Cross-Encoder inference

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Locate saved results
# --------------------------------------------------

RESULTS_DIR23 = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/reranking_eval_v1"
)

PROTOCOL_PATH23 = (
    RESULTS_DIR23 / "reranking_protocol_v1.json"
)

CANDIDATES_PATH23 = (
    RESULTS_DIR23 / "dev_late_candidates.jsonl"
)

PILOT_PATH23 = (
    RESULTS_DIR23 / "pilot_top5_predictions.jsonl"
)

PILOT_METRICS_PATH23 = (
    RESULTS_DIR23 / "pilot_top5_metrics.csv"
)

PILOT_MANIFEST_PATH23 = (
    RESULTS_DIR23 / "pilot_top5_manifest.json"
)

CHECKPOINT_DIR23 = (
    RESULTS_DIR23 / "dev_top5_checkpoints"
)

print("CELL 23 — CONSOLIDATED DEVELOPMENT EVALUATION")
print("=" * 60)


# --------------------------------------------------
# 2. Helpers
# --------------------------------------------------

def sha256_23(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def load_jsonl_23(path):

    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


# --------------------------------------------------
# 3. Verify frozen protocol and pilot
# --------------------------------------------------

protocol23 = json.loads(
    PROTOCOL_PATH23.read_text(encoding="utf-8")
)

pilot_manifest23 = json.loads(
    PILOT_MANIFEST_PATH23.read_text(
        encoding="utf-8"
    )
)

protocol_sha23 = sha256_23(PROTOCOL_PATH23)
candidate_sha23 = sha256_23(CANDIDATES_PATH23)

assert (
    candidate_sha23
    == protocol23["development"]["candidate_sha256"]
)

assert (
    sha256_23(PILOT_PATH23)
    == pilot_manifest23["result_files"][
        "predictions_sha256"
    ]
)

assert (
    sha256_23(PILOT_METRICS_PATH23)
    == pilot_manifest23["result_files"][
        "metrics_sha256"
    ]
)

assert protocol23["retrieval"]["rerank_top_k"] == 5

assert (
    pilot_manifest23["model"]["revision"]
    == protocol23["model"]["revision"]
)

print("Frozen protocol: VERIFIED")
print("Candidate source: VERIFIED")
print("Pilot files: VERIFIED")


# --------------------------------------------------
# 4. Load development data
# --------------------------------------------------

pools23 = load_jsonl_23(CANDIDATES_PATH23)
pilot23 = load_jsonl_23(PILOT_PATH23)

pilot_metrics23 = pd.read_csv(
    PILOT_METRICS_PATH23
).set_index("query_id")

assert len(pools23) == 60
assert len(pilot23) == 5

pilot_by_id23 = {
    record["query_id"]: record
    for record in pilot23
}

print("\nLoading 20 completed queries...")

# --------------------------------------------------
# 5. Reconstruct verified rankings
# --------------------------------------------------

rows23 = []

for index, pool in enumerate(pools23[:20]):

    query_id = pool["query_id"]
    gold_id = pool["source_id"]

    assert pool["partition"] == "dev"
    assert pool["query_type"] == "late"

    hybrid_ids = [
        candidate["document_id"]
        for candidate in pool["candidates"]
    ]

    assert len(hybrid_ids) == len(set(hybrid_ids))
    assert gold_id in hybrid_ids

    original_top5 = hybrid_ids[:5]

    # First five queries: previously saved pilot
    if index < 5:

        saved = pilot_by_id23[query_id]

        assert saved["source_id"] == gold_id
        assert saved["hybrid_ids"] == original_top5

        reranked = saved["reranked_candidates"]

        pilot_row = pilot_metrics23.loc[query_id]

        inference_seconds = float(
            pilot_row["inference_seconds"]
        )

        source = "pilot"

    # Queries 6–20: individual checkpoints
    else:

        path = (
            CHECKPOINT_DIR23 / f"{query_id}.json"
        )

        assert path.is_file(), path

        saved = json.loads(
            path.read_text(encoding="utf-8")
        )

        assert saved["query_id"] == query_id
        assert saved["source_id"] == gold_id

        assert saved["protocol_sha256"] == protocol_sha23
        assert saved["candidate_sha256"] == candidate_sha23

        assert saved["model_revision"] == (
            protocol23["model"]["revision"]
        )

        assert saved["hybrid_top5_ids"] == original_top5

        reranked = saved["reranked_top5"]

        inference_seconds = float(
            saved["inference_seconds"]
        )

        source = "checkpoint"

    # Verify the reranker did not add or remove articles
    reranked_ids = [
        row["document_id"]
        for row in reranked
    ]

    assert len(reranked_ids) == 5
    assert set(reranked_ids) == set(original_top5)

    assert all(
        np.isfinite(row["reranker_score"])
        for row in reranked
    )

    assert reranked == sorted(
        reranked,
        key=lambda row: (
            -row["reranker_score"],
            row["hybrid_rank"],
            row["document_id"],
        )
    )

    # Rerank Top-5; preserve the remaining Hybrid order
    full_reranked_ids = (
        reranked_ids + hybrid_ids[5:]
    )

    hybrid_rank = hybrid_ids.index(gold_id) + 1

    reranker_rank = (
        full_reranked_ids.index(gold_id) + 1
    )

    if reranker_rank < hybrid_rank:
        outcome = "IMPROVED"

    elif reranker_rank > hybrid_rank:
        outcome = "DEGRADED"

    else:
        outcome = "UNCHANGED"

    # Verify saved ranking information
    if source == "checkpoint":

        assert saved["hybrid_gold_rank"] == hybrid_rank
        assert saved["reranker_gold_rank"] == reranker_rank
        assert saved["outcome"] == outcome

    else:

        assert int(
            pilot_row["hybrid_gold_rank"]
        ) == hybrid_rank

        assert int(
            pilot_row["reranker_gold_rank"]
        ) == reranker_rank

        assert pilot_row["outcome"] == outcome

    rows23.append({
        "query_id": query_id,
        "hybrid_rank": hybrid_rank,
        "reranker_rank": reranker_rank,
        "outcome": outcome,
        "inference_seconds": inference_seconds,
        "source": source,
    })


# --------------------------------------------------
# 6. Create consolidated table
# --------------------------------------------------

combined_df23 = pd.DataFrame(rows23)

assert len(combined_df23) == 20
assert combined_df23["query_id"].is_unique

print("\nQUERY-LEVEL RESULTS")
print("-" * 60)

display(combined_df23)


# --------------------------------------------------
# 7. Calculate retrieval metrics
# --------------------------------------------------

def metrics_from_ranks23(ranks):

    ranks = np.asarray(
        ranks,
        dtype=float
    )

    assert np.isfinite(ranks).all()
    assert (ranks >= 1).all()

    return {
        "recall@1": float(
            np.mean(ranks <= 1)
        ),
        "recall@5": float(
            np.mean(ranks <= 5)
        ),
        "recall@10": float(
            np.mean(ranks <= 10)
        ),
        "mrr@10": float(
            np.mean(
                np.where(
                    ranks <= 10,
                    1.0 / ranks,
                    0.0,
                )
            )
        ),
    }


hybrid_metrics23 = metrics_from_ranks23(
    combined_df23["hybrid_rank"]
)

reranker_metrics23 = metrics_from_ranks23(
    combined_df23["reranker_rank"]
)

metrics_df23 = pd.DataFrame({
    "Hybrid RRF": hybrid_metrics23,
    "Cross-Encoder": reranker_metrics23,
})

metrics_df23["Difference"] = (
    metrics_df23["Cross-Encoder"]
    - metrics_df23["Hybrid RRF"]
)

print("\nCONSOLIDATED METRICS — 20 QUERIES")
print("=" * 60)

display(metrics_df23.round(4))


# --------------------------------------------------
# 8. Ranking outcomes and CPU performance
# --------------------------------------------------

print("\nRANKING OUTCOMES")
print("-" * 55)

for outcome in [
    "IMPROVED",
    "UNCHANGED",
    "DEGRADED",
]:

    count = int(
        (combined_df23["outcome"] == outcome).sum()
    )

    print(f"{outcome}: {count}")


print("\nCPU PERFORMANCE")
print("-" * 55)

print(
    "Completed queries:",
    len(combined_df23),
    "/ 60"
)

print(
    "Total measured inference:",
    f"{combined_df23['inference_seconds'].sum():.2f} sec"
)

print(
    "Mean inference per query:",
    f"{combined_df23['inference_seconds'].mean():.2f} sec"
)


# --------------------------------------------------
# 9. Verify against observed results
# --------------------------------------------------

assert np.isclose(
    hybrid_metrics23["recall@1"],
    0.70
)

assert np.isclose(
    reranker_metrics23["recall@1"],
    0.80
)

assert (
    (combined_df23["outcome"] == "IMPROVED").sum()
    == 3
)

assert (
    (combined_df23["outcome"] == "DEGRADED").sum()
    == 1
)

print("\nPreviously reported results: MATCHED")


# --------------------------------------------------
# 10. Save consolidated results
# --------------------------------------------------

SUMMARY_PATH23 = (
    RESULTS_DIR23 / "dev_top5_progress_20.csv"
)

METRICS_PATH23 = (
    RESULTS_DIR23 / "dev_top5_metrics_20.csv"
)

files23 = [
    (
        SUMMARY_PATH23,
        combined_df23.to_csv(
            index=False,
            lineterminator="\n",
            float_format="%.8f",
        ).encode("utf-8")
    ),
    (
        METRICS_PATH23,
        metrics_df23.to_csv(
            index=True,
            index_label="metric",
            lineterminator="\n",
            float_format="%.8f",
        ).encode("utf-8")
    ),
]

for path, content in files23:

    if path.exists():

        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )

        print("Existing file verified:", path.name)

    else:

        path.write_bytes(content)

        print("Saved:", path.name)

    assert path.read_bytes() == content


# --------------------------------------------------
# 11. Final verification
# --------------------------------------------------

print("\n" + "=" * 60)
print("CELL 23 VERIFICATION")
print("=" * 60)

print("Development queries: 20/60")
print("Pilot results verified: 5")
print("Individual checkpoints verified: 15")
print("Cross-Encoder inference: NOT EXECUTED")
print("Evaluation queries: NOT USED")
print("Original benchmark files: UNCHANGED")

print("\nSaved files:")
print(SUMMARY_PATH23)
print(METRICS_PATH23)

print("\nOK — Development progress 20/60 consolidated")

In [ ]:
# CELL 24 — Cross-Encoder Development Batch 21–25
# Notebook: 09_cross_encoder_reranking.ipynb
# Runtime: CPU

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Verify required variables
# --------------------------------------------------

required24 = [
    "run_dev_checkpoint_batch09",
    "dev_candidate_pools09",
    "CHECKPOINT_DIR19",
    "PROTOCOL_PATH18",
    "CANDIDATES_PATH09",
    "protocol_sha19",
    "RERANK_REVISION09",
]

missing24 = [
    name for name in required24
    if name not in globals()
]

assert not missing24, (
    "Missing variables: " + ", ".join(missing24)
)

assert len(dev_candidate_pools09) == 60
assert RERANK_TOP_K09 == 5

print("CELL 24 — DEVELOPMENT BATCH 21–25")
print("=" * 60)

# --------------------------------------------------
# 2. Verify frozen protocol
# --------------------------------------------------

protocol_bytes24 = PROTOCOL_PATH18.read_bytes()

protocol_sha24 = hashlib.sha256(
    protocol_bytes24
).hexdigest()

protocol24 = json.loads(protocol_bytes24)

candidate_sha24 = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

assert protocol_sha24 == protocol_sha19

assert candidate_sha24 == (
    protocol24["development"]["candidate_sha256"]
)

assert protocol24["retrieval"]["rerank_top_k"] == 5

assert protocol24["model"]["revision"] == (
    RERANK_REVISION09
)

print("Frozen protocol: VERIFIED")
print("Candidate dataset: VERIFIED")

# --------------------------------------------------
# 3. Verify previous progress
# --------------------------------------------------

PROGRESS_PATH24 = (
    RERANK_DIR09 / "dev_top5_progress_20.csv"
)

assert PROGRESS_PATH24.is_file()

previous_progress24 = pd.read_csv(
    PROGRESS_PATH24
)

assert len(previous_progress24) == 20

assert previous_progress24["query_id"].tolist() == [
    q["query_id"]
    for q in dev_candidate_pools09[:20]
]

# Pilot (1–5) is saved separately.
assert PILOT_PREDICTIONS_PATH09.is_file()

# Queries 6–20 must have checkpoints.
for q in dev_candidate_pools09[5:20]:

    checkpoint = (
        CHECKPOINT_DIR19 / f"{q['query_id']}.json"
    )

    assert checkpoint.is_file(), checkpoint

print("Previous 20 development queries: VERIFIED")

# --------------------------------------------------
# 4. Select next development queries
# --------------------------------------------------

START24 = 20
END24 = 25

selected24 = dev_candidate_pools09[
    START24:END24
]

expected_ids24 = [
    f"late_{i:03d}"
    for i in range(21, 26)
]

assert [
    q["query_id"]
    for q in selected24
] == expected_ids24

assert all(
    q["partition"] == "dev"
    and q["query_type"] == "late"
    for q in selected24
)

print("\nQUERIES TO PROCESS")
print("-" * 55)

for query_id in expected_ids24:
    print(query_id)

# --------------------------------------------------
# 5. Run checkpointed Cross-Encoder inference
# --------------------------------------------------

batch_records24 = run_dev_checkpoint_batch09(
    start_index=START24,
    end_index=END24,
)

assert len(batch_records24) == 5

assert [
    r["query_id"]
    for r in batch_records24
] == expected_ids24

# --------------------------------------------------
# 6. Verify saved checkpoints
# --------------------------------------------------

for record in batch_records24:

    checkpoint_path = (
        CHECKPOINT_DIR19
        / f"{record['query_id']}.json"
    )

    assert checkpoint_path.is_file()

    saved = json.loads(
        checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    assert saved == record

    assert saved["protocol_sha256"] == protocol_sha24

    assert saved["candidate_sha256"] == candidate_sha24

    assert saved["model_revision"] == (
        RERANK_REVISION09
    )

print("\nAll 5 checkpoints verified.")

# --------------------------------------------------
# 7. Build query-level results
# --------------------------------------------------

batch_df24 = pd.DataFrame([
    {
        "query_id": r["query_id"],
        "hybrid_rank": r["hybrid_gold_rank"],
        "reranker_rank": r["reranker_gold_rank"],
        "outcome": r["outcome"],
        "inference_seconds": r["inference_seconds"],
    }
    for r in batch_records24
])

assert len(batch_df24) == 5
assert batch_df24["query_id"].is_unique

print("\nQUERY-LEVEL RESULTS")
print("-" * 55)

display(batch_df24.round(3))

# --------------------------------------------------
# 8. Calculate batch metrics
# --------------------------------------------------

def calculate_metrics24(ranks):

    ranks = np.asarray(ranks, dtype=float)

    return {
        "recall@1": float(
            np.mean(ranks <= 1)
        ),
        "recall@5": float(
            np.mean(ranks <= 5)
        ),
        "recall@10": float(
            np.mean(ranks <= 10)
        ),
        "mrr@10": float(
            np.mean(
                np.where(
                    ranks <= 10,
                    1.0 / ranks,
                    0.0
                )
            )
        ),
    }


hybrid_metrics24 = calculate_metrics24(
    batch_df24["hybrid_rank"]
)

reranker_metrics24 = calculate_metrics24(
    batch_df24["reranker_rank"]
)

comparison24 = pd.DataFrame({
    "Hybrid RRF": hybrid_metrics24,
    "Cross-Encoder": reranker_metrics24,
})

comparison24["Difference"] = (
    comparison24["Cross-Encoder"]
    - comparison24["Hybrid RRF"]
)

print("\nBATCH METRICS")
print("=" * 55)

display(comparison24.round(4))

# --------------------------------------------------
# 9. Ranking outcomes
# --------------------------------------------------

print("\nRANKING OUTCOMES")
print("-" * 55)

for outcome in [
    "IMPROVED",
    "UNCHANGED",
    "DEGRADED",
]:

    count = int(
        (batch_df24["outcome"] == outcome).sum()
    )

    print(f"{outcome}: {count}")

# --------------------------------------------------
# 10. CPU performance
# --------------------------------------------------

print("\nCPU PERFORMANCE")
print("-" * 55)

print(
    "Mean inference per query:",
    f"{batch_df24['inference_seconds'].mean():.2f} sec"
)

print(
    "Total measured inference:",
    f"{batch_df24['inference_seconds'].sum():.2f} sec"
)

# --------------------------------------------------
# 11. Final verification
# --------------------------------------------------

assert all(
    (
        CHECKPOINT_DIR19 / f"{q['query_id']}.json"
    ).is_file()
    for q in selected24
)

print("\n" + "=" * 60)
print("CELL 24 VERIFICATION")
print("=" * 60)

print("Development queries completed: 25/60")
print("Current batch: late_021 to late_025")
print("Checkpoints: VERIFIED")
print("Frozen protocol: VERIFIED")
print("Evaluation queries: NOT USED")
print("Original benchmark: UNCHANGED")

print("\nOK — Development batch 21–25 completed")

In [ ]:
# CELL 25 — Automatic Development Reranking
# Resume from saved checkpoints — CPU

import json
import hashlib
from pathlib import Path

# Number of new queries per execution
MAX_NEW_QUERIES = 10

assert len(dev_candidate_pools09) == 60
assert RERANK_TOP_K09 == 5
assert PILOT_PREDICTIONS_PATH09.is_file()
assert callable(run_dev_checkpoint_batch09)

protocol = json.loads(
    PROTOCOL_PATH18.read_text(encoding="utf-8")
)

protocol_sha = hashlib.sha256(
    PROTOCOL_PATH18.read_bytes()
).hexdigest()

candidate_sha = hashlib.sha256(
    CANDIDATES_PATH09.read_bytes()
).hexdigest()

assert candidate_sha == protocol["development"]["candidate_sha256"]
assert protocol["model"]["revision"] == RERANK_REVISION09
assert protocol["retrieval"]["rerank_top_k"] == 5

# Verify existing checkpoints before continuing
for q in dev_candidate_pools09[5:]:
    path = CHECKPOINT_DIR19 / f"{q['query_id']}.json"

    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))

        assert saved["query_id"] == q["query_id"]
        assert saved["source_id"] == q["source_id"]
        assert saved["protocol_sha256"] == protocol_sha
        assert saved["candidate_sha256"] == candidate_sha
        assert saved["model_revision"] == RERANK_REVISION09
        assert saved["rerank_top_k"] == 5
        assert saved["hybrid_top5_ids"] == [
            c["document_id"] for c in q["candidates"][:5]
        ]

# Identify missing queries
pending = [
    i for i in range(5, 60)
    if not (
        CHECKPOINT_DIR19
        / f"{dev_candidate_pools09[i]['query_id']}.json"
    ).exists()
]

print("AUTOMATIC DEVELOPMENT RERANKING")
print("=" * 60)
print(f"Completed: {60 - len(pending)}/60")
print(f"Remaining: {len(pending)}")

if pending:
    start = pending[0]
    end = min(start + MAX_NEW_QUERIES, 60)

    print(
        "\nProcessing:",
        dev_candidate_pools09[start]["query_id"],
        "to",
        dev_candidate_pools09[end - 1]["query_id"]
    )

    records = run_dev_checkpoint_batch09(start, end)

    print(f"\nBatch finished: {len(records)} queries")

# Count progress after execution
remaining = [
    q["query_id"]
    for q in dev_candidate_pools09[5:]
    if not (
        CHECKPOINT_DIR19 / f"{q['query_id']}.json"
    ).exists()
]

completed = 60 - len(remaining)

print("\n" + "=" * 60)
print("FINAL PROGRESS")
print("=" * 60)
print(f"Completed: {completed}/60")
print(f"Remaining: {len(remaining)}")
print("Evaluation queries: NOT USED")

if not remaining:
    print("\nALL 60 DEVELOPMENT QUERIES COMPLETED!")
    print("Ready for final development evaluation.")
else:
    print("\nRun this SAME cell again to continue.")

print("\nOK — Progress checkpointed")

In [ ]:
# CELL 26 — Final Cross-Encoder Development Evaluation (60/60)
# No inference — Read saved checkpoints only

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. File paths
# --------------------------------------------------

ROOT26 = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/reranking_eval_v1"
)

PROTOCOL26 = ROOT26 / "reranking_protocol_v1.json"
CANDIDATES26 = ROOT26 / "dev_late_candidates.jsonl"
PILOT26 = ROOT26 / "pilot_top5_predictions.jsonl"
PILOT_METRICS26 = ROOT26 / "pilot_top5_metrics.csv"
PILOT_MANIFEST26 = ROOT26 / "pilot_top5_manifest.json"

CHECKPOINTS26 = ROOT26 / "dev_top5_checkpoints"

def sha26(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def read_jsonl26(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

print("CELL 26 — FINAL DEVELOPMENT EVALUATION")
print("=" * 60)

# --------------------------------------------------
# 2. Verify experiment provenance
# --------------------------------------------------

protocol26 = json.loads(
    PROTOCOL26.read_text(encoding="utf-8")
)

manifest26 = json.loads(
    PILOT_MANIFEST26.read_text(encoding="utf-8")
)

protocol_hash26 = sha26(PROTOCOL26)
candidate_hash26 = sha26(CANDIDATES26)

assert candidate_hash26 == (
    protocol26["development"]["candidate_sha256"]
)

assert protocol26["retrieval"]["rerank_top_k"] == 5

assert sha26(PILOT26) == (
    manifest26["result_files"]["predictions_sha256"]
)

assert sha26(PILOT_METRICS26) == (
    manifest26["result_files"]["metrics_sha256"]
)

assert manifest26["model"]["revision"] == (
    protocol26["model"]["revision"]
)

print("Frozen protocol: VERIFIED")
print("Candidate source: VERIFIED")
print("Pilot results: VERIFIED")

# --------------------------------------------------
# 3. Load all development queries
# --------------------------------------------------

pools26 = read_jsonl26(CANDIDATES26)
pilot26 = read_jsonl26(PILOT26)

pilot_metrics26 = pd.read_csv(
    PILOT_METRICS26
).set_index("query_id")

assert len(pools26) == 60
assert len(pilot26) == 5
assert len(pilot_metrics26) == 5

pilot_by_id26 = {
    p["query_id"]: p
    for p in pilot26
}

# --------------------------------------------------
# 4. Ensure all 55 checkpoints exist
# --------------------------------------------------

missing26 = [
    q["query_id"]
    for q in pools26[5:]
    if not (
        CHECKPOINTS26 / f"{q['query_id']}.json"
    ).is_file()
]

if missing26:
    raise RuntimeError(
        f"Only {60 - len(missing26)}/60 completed. "
        f"Run Cell 25 again. Missing: {missing26}"
    )

print("Development queries: 60/60")
print("Saved checkpoints: 55/55")

# --------------------------------------------------
# 5. Reconstruct rankings
# --------------------------------------------------

rows26 = []

for i, pool in enumerate(pools26):

    qid = pool["query_id"]
    gold = pool["source_id"]

    assert qid == f"late_{i+1:03d}"
    assert pool["partition"] == "dev"
    assert pool["query_type"] == "late"

    hybrid_ids = [
        c["document_id"]
        for c in pool["candidates"]
    ]

    assert len(hybrid_ids) == len(set(hybrid_ids))
    assert gold in hybrid_ids

    original_top5 = hybrid_ids[:5]
    assert len(original_top5) == 5

    if i < 5:

        record = pilot_by_id26[qid]

        assert record["source_id"] == gold
        assert record["hybrid_ids"] == original_top5

        reranked = record["reranked_candidates"]

        elapsed = float(
            pilot_metrics26.loc[
                qid, "inference_seconds"
            ]
        )

        source = "pilot"

    else:

        path = CHECKPOINTS26 / f"{qid}.json"

        record = json.loads(
            path.read_text(encoding="utf-8")
        )

        assert record["query_id"] == qid
        assert record["source_id"] == gold

        assert record["protocol_sha256"] == (
            protocol_hash26
        )

        assert record["candidate_sha256"] == (
            candidate_hash26
        )

        assert record["model_revision"] == (
            protocol26["model"]["revision"]
        )

        assert record["hybrid_top5_ids"] == (
            original_top5
        )

        assert record["rerank_top_k"] == 5

        reranked = record["reranked_top5"]

        elapsed = float(
            record["inference_seconds"]
        )

        source = "checkpoint"

    # Verify exactly the same candidate set
    reranked_ids = [
        r["document_id"]
        for r in reranked
    ]

    assert len(reranked_ids) == 5
    assert set(reranked_ids) == set(original_top5)

    for r in reranked:

        assert np.isfinite(r["reranker_score"])

        assert r["hybrid_rank"] == (
            original_top5.index(r["document_id"]) + 1
        )

    # Verify deterministic sorting
    assert reranked == sorted(
        reranked,
        key=lambda r: (
            -r["reranker_score"],
            r["hybrid_rank"],
            r["document_id"],
        )
    )

    # Top-5 reranking; untouched Hybrid tail
    final_ids = reranked_ids + hybrid_ids[5:]

    hybrid_rank = hybrid_ids.index(gold) + 1
    reranker_rank = final_ids.index(gold) + 1

    if reranker_rank < hybrid_rank:
        outcome = "IMPROVED"

    elif reranker_rank > hybrid_rank:
        outcome = "DEGRADED"

    else:
        outcome = "UNCHANGED"

    # Compare reconstructed ranks with saved ranks
    if source == "checkpoint":

        assert record["hybrid_gold_rank"] == hybrid_rank
        assert record["reranker_gold_rank"] == reranker_rank
        assert record["outcome"] == outcome

    else:

        saved_pilot_row = pilot_metrics26.loc[qid]

        assert int(
            saved_pilot_row["hybrid_gold_rank"]
        ) == hybrid_rank

        assert int(
            saved_pilot_row["reranker_gold_rank"]
        ) == reranker_rank

        assert saved_pilot_row["outcome"] == outcome

    rows26.append({
        "query_id": qid,
        "hybrid_rank": hybrid_rank,
        "reranker_rank": reranker_rank,
        "outcome": outcome,
        "inference_seconds": elapsed,
        "source": source,
    })

results26 = pd.DataFrame(rows26)

assert len(results26) == 60
assert results26["query_id"].is_unique

print("\nAll rankings reconstructed and verified.")

# --------------------------------------------------
# 6. Calculate final development metrics
# --------------------------------------------------

def compute_metrics26(ranks):

    ranks = np.asarray(ranks, dtype=float)

    return {
        "recall@1": float(np.mean(ranks <= 1)),
        "recall@5": float(np.mean(ranks <= 5)),
        "recall@10": float(np.mean(ranks <= 10)),
        "mrr@10": float(
            np.mean(
                np.where(
                    ranks <= 10,
                    1.0 / ranks,
                    0.0
                )
            )
        ),
    }

hybrid26 = compute_metrics26(
    results26["hybrid_rank"]
)

reranker26 = compute_metrics26(
    results26["reranker_rank"]
)

# Verify the frozen Hybrid baseline
assert np.isclose(hybrid26["recall@1"], 0.60)
assert np.isclose(hybrid26["recall@5"], 0.85)
assert np.isclose(hybrid26["recall@10"], 59 / 60)

metrics26 = pd.DataFrame({
    "Hybrid RRF": hybrid26,
    "Cross-Encoder": reranker26,
})

metrics26["Difference"] = (
    metrics26["Cross-Encoder"]
    - metrics26["Hybrid RRF"]
)

print("\nFINAL DEVELOPMENT METRICS — 60 QUERIES")
print("=" * 60)

display(metrics26.round(4))

# --------------------------------------------------
# 7. Ranking outcomes and runtime
# --------------------------------------------------

outcomes26 = {
    name: int(
        (results26["outcome"] == name).sum()
    )
    for name in ["IMPROVED", "UNCHANGED", "DEGRADED"]
}

assert sum(outcomes26.values()) == 60

total_seconds26 = float(
    results26["inference_seconds"].sum()
)

print("\nRANKING OUTCOMES")

for name, count in outcomes26.items():
    print(f"{name}: {count}")

print("\nCPU PERFORMANCE")

print(
    f"Total inference: {total_seconds26 / 60:.2f} minutes"
)

print(
    f"Mean per query: {total_seconds26 / 60:.2f} seconds"
)

# --------------------------------------------------
# 8. Prepare final files
# --------------------------------------------------

PREDICTIONS_OUT26 = (
    ROOT26 / "dev_top5_predictions_60.csv"
)

METRICS_OUT26 = (
    ROOT26 / "dev_top5_metrics_60.csv"
)

REPORT_OUT26 = (
    ROOT26 / "dev_top5_report_60.json"
)

predictions_bytes26 = results26.to_csv(
    index=False,
    float_format="%.8f",
    lineterminator="\n",
).encode("utf-8")

metrics_bytes26 = metrics26.to_csv(
    index=True,
    index_label="metric",
    float_format="%.8f",
    lineterminator="\n",
).encode("utf-8")

report26 = {
    "experiment": "cross_encoder_development_top5",
    "status": "development_complete",
    "partition": "dev",
    "query_type": "late",
    "n_queries": 60,
    "model": protocol26["model"],
    "retrieval": protocol26["retrieval"],
    "protocol_sha256": protocol_hash26,
    "candidate_sha256": candidate_hash26,
    "metrics": {
        "hybrid_rrf": hybrid26,
        "cross_encoder": reranker26,
    },
    "ranking_outcomes": outcomes26,
    "cpu_inference": {
        "total_seconds": total_seconds26,
        "mean_seconds_per_query": total_seconds26 / 60,
    },
    "result_files": {
        "predictions": PREDICTIONS_OUT26.name,
        "predictions_sha256": hashlib.sha256(
            predictions_bytes26
        ).hexdigest(),
        "metrics": METRICS_OUT26.name,
        "metrics_sha256": hashlib.sha256(
            metrics_bytes26
        ).hexdigest(),
        "individual_checkpoints": CHECKPOINTS26.name,
    },
    "limitations": protocol26["limitations"],
    "evaluation_queries_used": False,
}

report_bytes26 = (
    json.dumps(
        report26,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
).encode("utf-8")

# --------------------------------------------------
# 9. Save safely
# --------------------------------------------------

for path, content in [
    (PREDICTIONS_OUT26, predictions_bytes26),
    (METRICS_OUT26, metrics_bytes26),
    (REPORT_OUT26, report_bytes26),
]:

    if path.exists():

        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )

        print("Verified:", path.name)

    else:

        path.write_bytes(content)

        print("Saved:", path.name)

    assert path.read_bytes() == content

# --------------------------------------------------
# 10. Final verification
# --------------------------------------------------

print("\n" + "=" * 60)
print("CELL 26 VERIFICATION")
print("=" * 60)

print("Development queries: 60/60")
print("Pilot predictions: 5/5")
print("Individual checkpoints: 55/55")
print("Hybrid baseline: VERIFIED")
print("Cross-Encoder inference: NOT EXECUTED")
print("Evaluation queries: NOT USED")
print("Frozen protocol: UNCHANGED")
print("Results: SAVED")

print("\nOK — Final development evaluation completed")

In [ ]:
# CELL 27 — Frozen Evaluation Preparation
# Notebook: 09_cross_encoder_reranking.ipynb
# CPU — No inference

import json
import hashlib
from pathlib import Path
from collections import Counter

# ------------------------------------
# 1. Paths
# ------------------------------------

ROOT27 = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/reranking_eval_v1"
)

PROTOCOL27 = ROOT27 / "reranking_protocol_v1.json"
DEV_REPORT27 = ROOT27 / "dev_top5_report_60.json"

QUERY_SOURCE27 = Path(EVAL_DIR) / "queries.jsonl"

EVAL_OUTPUT27 = ROOT27 / "frozen_eval_v1"
EVAL_OUTPUT27.mkdir(parents=True, exist_ok=True)

FROZEN_QUERIES27 = (
    EVAL_OUTPUT27 / "evaluation_queries_frozen.jsonl"
)

EVAL_SETUP27 = (
    EVAL_OUTPUT27 / "evaluation_setup.json"
)

def sha256_27(data):
    return hashlib.sha256(data).hexdigest()

def save_verified27(path, content):

    if path.exists():
        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )
        print("Verified:", path.name)

    else:
        path.write_bytes(content)
        print("Saved:", path.name)

    assert path.read_bytes() == content


print("CELL 27 — FROZEN EVALUATION PREPARATION")
print("=" * 60)

# ------------------------------------
# 2. Verify completed development
# ------------------------------------

assert PROTOCOL27.is_file()
assert DEV_REPORT27.is_file()
assert QUERY_SOURCE27.is_file()

protocol_bytes27 = PROTOCOL27.read_bytes()
dev_bytes27 = DEV_REPORT27.read_bytes()

protocol27 = json.loads(protocol_bytes27)
dev_report27 = json.loads(dev_bytes27)

assert dev_report27["status"] == "development_complete"
assert dev_report27["n_queries"] == 60

assert dev_report27["protocol_sha256"] == (
    sha256_27(protocol_bytes27)
)

assert dev_report27["evaluation_queries_used"] is False

assert dev_report27["ranking_outcomes"] == {
    "IMPROVED": 13,
    "UNCHANGED": 44,
    "DEGRADED": 3,
}

assert protocol27["retrieval"]["rerank_top_k"] == 5

print("Development benchmark: VERIFIED")
print("Frozen protocol: VERIFIED")

# ------------------------------------
# 3. Read original frozen queries
# ------------------------------------

source_bytes27 = QUERY_SOURCE27.read_bytes()

source_rows27 = [
    json.loads(line)
    for line in source_bytes27.decode("utf-8").splitlines()
    if line.strip()
]

evaluation_queries27 = [
    q for q in source_rows27
    if q["partition"] in ("evaluation", "eval")
]

assert len(evaluation_queries27) == 480, (
    f"Expected 480 evaluation queries, "
    f"found {len(evaluation_queries27)}"
)

query_counts27 = Counter(
    q["query_type"] for q in evaluation_queries27
)

assert query_counts27 == {
    "lead": 240,
    "late": 240,
}

query_ids27 = [
    q["query_id"] for q in evaluation_queries27
]

assert len(query_ids27) == len(set(query_ids27))

assert all(
    q["query"] and q["source_id"]
    for q in evaluation_queries27
)

print("\nFROZEN EVALUATION QUERIES")
print("-" * 55)

print("Total:", len(evaluation_queries27))
print("Lead queries:", query_counts27["lead"])
print("Late queries:", query_counts27["late"])
print("Unique query IDs: VERIFIED")

# ------------------------------------
# 4. Freeze exact evaluation records
# ------------------------------------

frozen_bytes27 = "".join(
    json.dumps(
        q,
        ensure_ascii=False,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
    for q in evaluation_queries27
).encode("utf-8")

frozen_sha27 = sha256_27(frozen_bytes27)

save_verified27(
    FROZEN_QUERIES27,
    frozen_bytes27
)

# ------------------------------------
# 5. Create evaluation setup
# ------------------------------------

setup27 = {
    "experiment": "cross_encoder_frozen_evaluation_v1",
    "status": "prepared",

    "protocol_sha256": sha256_27(protocol_bytes27),
    "development_report_sha256": sha256_27(dev_bytes27),

    "original_queries_sha256": sha256_27(source_bytes27),
    "frozen_queries_sha256": frozen_sha27,

    "evaluation": {
        "total_queries": 480,
        "lead_queries": 240,
        "late_queries": 240,
        "hyperparameter_tuning_allowed": False,
    },

    "model": protocol27["model"],
    "retrieval": protocol27["retrieval"],
    "metrics": protocol27["metrics"],

    "execution": {
        "checkpointing": True,
        "rerank_top_k": 5,
        "use_existing_hybrid_retrieval": True,
        "retrain_models": False,
    },

    "limitations": protocol27["limitations"],
}

setup_bytes27 = (
    json.dumps(
        setup27,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

save_verified27(
    EVAL_SETUP27,
    setup_bytes27
)

# ------------------------------------
# 6. Final verification
# ------------------------------------

print("\n" + "=" * 60)
print("CELL 27 VERIFICATION")
print("=" * 60)

print("Development evaluation: COMPLETE")
print("Evaluation queries:", len(evaluation_queries27))
print("Model configuration: FROZEN")
print("Evaluation inference: NOT EXECUTED")
print("Evaluation metrics: NOT INSPECTED")
print("Original benchmark: UNCHANGED")

print("\nFrozen queries SHA256:")
print(frozen_sha27)

print("\nOutput directory:")
print(EVAL_OUTPUT27)

print("\nOK — Frozen evaluation prepared")

In [ ]:
# CELL 28 — Frozen Evaluation Hybrid Candidate Pools
# CPU | Resume | Per-query checkpoints | No Cross-Encoder inference

import json
import hashlib
import re
import time
from pathlib import Path
from collections import Counter

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

MAX_NEW_QUERIES28 = 400

ROOT28 = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/reranking_eval_v1/frozen_eval_v1"
)

QUERY_PATH28 = ROOT28 / "evaluation_queries_frozen.jsonl"
SETUP_PATH28 = ROOT28 / "evaluation_setup.json"

PROTOCOL_PATH28 = ROOT28.parent / "reranking_protocol_v1.json"
CANDIDATE_DIR28 = ROOT28 / "hybrid_candidate_checkpoints"

CANDIDATE_FILE28 = ROOT28 / "evaluation_hybrid_candidates.jsonl"
CANDIDATE_MANIFEST28 = ROOT28 / "hybrid_candidates_manifest.json"

CANDIDATE_DIR28.mkdir(parents=True, exist_ok=True)

assert callable(search_hybrid09)
assert len(dev_candidate_pools09) == 60
assert len(doc_by_id09) == 5250
assert 1 <= MAX_NEW_QUERIES28 <= 480

def sha28(data):
    return hashlib.sha256(data).hexdigest()

def read_jsonl28(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(s) for s in f if s.strip()]

def json_bytes28(value):
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            allow_nan=False,
        ) + "\n"
    ).encode("utf-8")

def save_once28(path, content):
    if path.exists():
        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )
    else:
        temporary = path.with_name(path.name + ".tmp")
        temporary.write_bytes(content)
        temporary.replace(path)

    assert path.read_bytes() == content

# --------------------------------------------------
# 2. Verify frozen input files
# --------------------------------------------------

query_bytes28 = QUERY_PATH28.read_bytes()
setup_bytes28 = SETUP_PATH28.read_bytes()
protocol_bytes28 = PROTOCOL_PATH28.read_bytes()

query_sha28 = sha28(query_bytes28)
setup_sha28 = sha28(setup_bytes28)
protocol_sha28 = sha28(protocol_bytes28)

setup28 = json.loads(setup_bytes28)
protocol28 = json.loads(protocol_bytes28)

assert setup28["frozen_queries_sha256"] == query_sha28
assert setup28["protocol_sha256"] == protocol_sha28

assert protocol28["retrieval"]["rerank_top_k"] == 5
assert protocol28["retrieval"]["bm25_candidate_k"] == 50
assert protocol28["retrieval"]["dense_candidate_k"] == 50
assert protocol28["retrieval"]["rrf_k"] == 60

queries28 = read_jsonl28(QUERY_PATH28)

assert len(queries28) == 480
assert len({q["query_id"] for q in queries28}) == 480

assert Counter(q["query_type"] for q in queries28) == {
    "lead": 240,
    "late": 240,
}

assert all(q["partition"] in ("eval", "evaluation")
           for q in queries28)

print("CELL 28 — FROZEN HYBRID CANDIDATE GENERATION")
print("=" * 60)

print("Frozen queries: VERIFIED")
print("Frozen protocol: VERIFIED")
print("Corpus documents:", len(doc_by_id09))
print("Queries:", len(queries28))

# --------------------------------------------------
# 3. Normalize retrieval output
# --------------------------------------------------

def normalize_candidates28(raw_results):

    candidates = []

    for rank, item in enumerate(raw_results, start=1):

        if not isinstance(item, dict):
            raise TypeError(
                "Unexpected search_hybrid09 output type: "
                f"{type(item)}"
            )

        document_id = next(
            (
                item[key]
                for key in ("document_id", "doc_id", "id")
                if item.get(key) is not None
            ),
            None,
        )

        if document_id is None:
            raise KeyError(
                "No document ID found. "
                f"Available fields: {list(item.keys())}"
            )

        document_id = str(document_id)

        assert document_id in doc_by_id09

        score = item.get(
            "rrf_score",
            item.get("score")
        )

        bm25_rank = item.get("bm25_rank")
        dense_rank = item.get("dense_rank")

        candidates.append({
            "rank": rank,
            "document_id": document_id,
            "rrf_score": (
                float(score)
                if score is not None else None
            ),
            "bm25_rank": (
                int(bm25_rank)
                if bm25_rank is not None else None
            ),
            "dense_rank": (
                int(dense_rank)
                if dense_rank is not None else None
            ),
        })

    ids = [c["document_id"] for c in candidates]

    assert len(ids) == len(set(ids))
    assert 5 <= len(candidates) <= 100

    return candidates

# --------------------------------------------------
# 4. Preflight using development data
# --------------------------------------------------

reference28 = dev_candidate_pools09[0]

preflight_raw28 = search_hybrid09(
    reference28["query"],
    k=100,
)

preflight28 = normalize_candidates28(preflight_raw28)

assert [
    c["document_id"] for c in preflight28[:10]
] == [
    c["document_id"]
    for c in reference28["candidates"][:10]
], "Hybrid retrieval does not match saved development candidates."

print("Hybrid retrieval consistency: VERIFIED")

# --------------------------------------------------
# 5. Verify checkpoint metadata
# --------------------------------------------------

def checkpoint_path28(q):
    qid = q["query_id"]

    assert re.fullmatch(r"(lead|late)_\d{3}", qid)

    return CANDIDATE_DIR28 / f"{qid}.json"


def verify_checkpoint28(q, record):

    assert record["query_id"] == q["query_id"]
    assert record["query_type"] == q["query_type"]
    assert record["source_id"] == q["source_id"]
    assert record["query"] == q["query"]

    assert record["protocol_sha256"] == protocol_sha28
    assert record["frozen_queries_sha256"] == query_sha28
    assert record["evaluation_setup_sha256"] == setup_sha28

    candidates = record["candidates"]

    assert 5 <= len(candidates) <= 100

    ids = [c["document_id"] for c in candidates]

    assert len(ids) == len(set(ids))
    assert all(doc_id in doc_by_id09 for doc_id in ids)

    assert [c["rank"] for c in candidates] == list(
        range(1, len(candidates) + 1)
    )

# Verify all previously saved checkpoints
pending28 = []

for q in queries28:

    path = checkpoint_path28(q)

    if path.exists():

        record = json.loads(
            path.read_text(encoding="utf-8")
        )

        verify_checkpoint28(q, record)

    else:
        pending28.append(q)

completed_before28 = 480 - len(pending28)

print("\nCURRENT PROGRESS")
print("-" * 55)
print(f"Completed: {completed_before28}/480")
print(f"Remaining: {len(pending28)}")

# --------------------------------------------------
# 6. Generate next batch
# --------------------------------------------------

selected28 = pending28[:MAX_NEW_QUERIES28]

start_time28 = time.perf_counter()

for index, q in enumerate(selected28, start=1):

    qid = q["query_id"]
    path = checkpoint_path28(q)

    raw = search_hybrid09(
        q["query"],
        k=100,
    )

    candidates = normalize_candidates28(raw)

    record = {
        "query_id": qid,
        "query_type": q["query_type"],
        "partition": "evaluation",
        "source_id": q["source_id"],
        "source_split": q.get("source_split"),
        "query": q["query"],
        "protocol_sha256": protocol_sha28,
        "frozen_queries_sha256": query_sha28,
        "evaluation_setup_sha256": setup_sha28,
        "candidates": candidates,
    }

    verify_checkpoint28(q, record)

    save_once28(
        path,
        json_bytes28(record)
    )

    if index == 1 or index % 10 == 0 or index == len(selected28):
        print(
            f"[{index}/{len(selected28)}] "
            f"{qid} | "
            f"{len(candidates)} candidates | SAVED",
            flush=True
        )

elapsed28 = time.perf_counter() - start_time28

# --------------------------------------------------
# 7. Verify progress
# --------------------------------------------------

remaining28 = [
    q
    for q in queries28
    if not checkpoint_path28(q).is_file()
]

completed28 = 480 - len(remaining28)

print("\nPROGRESS")
print("=" * 60)

print(f"Completed: {completed28}/480")
print(f"Remaining: {len(remaining28)}")
print(f"Current batch elapsed: {elapsed28 / 60:.2f} min")

# --------------------------------------------------
# 8. Finalize only when all 480 are complete
# --------------------------------------------------

if not remaining28:

    all_records28 = []

    for q in queries28:

        record = json.loads(
            checkpoint_path28(q).read_text(
                encoding="utf-8"
            )
        )

        verify_checkpoint28(q, record)

        all_records28.append(record)

    candidate_bytes28 = "".join(
        json.dumps(
            r,
            ensure_ascii=False,
            sort_keys=True,
            allow_nan=False,
        ) + "\n"
        for r in all_records28
    ).encode("utf-8")

    candidate_sha28 = sha28(candidate_bytes28)

    save_once28(
        CANDIDATE_FILE28,
        candidate_bytes28,
    )

    manifest28 = {
        "experiment": "frozen_evaluation_hybrid_candidates_v1",
        "status": "complete",
        "n_queries": 480,
        "lead_queries": 240,
        "late_queries": 240,
        "rerank_top_k": 5,
        "retrieval": protocol28["retrieval"],
        "model": protocol28["model"],
        "protocol_sha256": protocol_sha28,
        "evaluation_setup_sha256": setup_sha28,
        "frozen_queries_sha256": query_sha28,
        "candidate_file": CANDIDATE_FILE28.name,
        "candidate_file_sha256": candidate_sha28,
        "candidate_counts": {
            "minimum": min(
                len(r["candidates"])
                for r in all_records28
            ),
            "maximum": max(
                len(r["candidates"])
                for r in all_records28
            ),
        },
        "evaluation_metrics_computed": False,
        "cross_encoder_inference_executed": False,
    }

    save_once28(
        CANDIDATE_MANIFEST28,
        json_bytes28(manifest28),
    )

    print("\nALL 480 CANDIDATE POOLS COMPLETED!")

    print("Candidate SHA256:", candidate_sha28)

    print("\nSaved:")
    print(CANDIDATE_FILE28)
    print(CANDIDATE_MANIFEST28)

    print("\nOK — Frozen evaluation candidates ready")

else:

    print("\nRun this SAME Cell 28 again to continue.")
    print("Previously saved queries will be skipped.")

print("\nCross-Encoder inference: NOT EXECUTED")
print("Evaluation metrics: NOT COMPUTED")
print("Original benchmark: UNCHANGED")

In [ ]:
# CELL 29 — Automatic Frozen Cross-Encoder Evaluation
# Notebook: 09_cross_encoder_reranking.ipynb
# CPU | Top-5 | Resume | Checkpoints
# No training — No evaluation metrics computed

import json
import time
import hashlib
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

MAX_NEW_QUERIES29 = 480

ROOT29 = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/reranking_eval_v1/frozen_eval_v1"
)

QUERY_FILE29 = ROOT29 / "evaluation_queries_frozen.jsonl"
SETUP_FILE29 = ROOT29 / "evaluation_setup.json"

CANDIDATE_FILE29 = (
    ROOT29 / "evaluation_hybrid_candidates.jsonl"
)

CANDIDATE_MANIFEST29 = (
    ROOT29 / "hybrid_candidates_manifest.json"
)

PROTOCOL_FILE29 = (
    ROOT29.parent / "reranking_protocol_v1.json"
)

CHECKPOINT_DIR29 = ROOT29 / "cross_encoder_checkpoints"
CHECKPOINT_DIR29.mkdir(parents=True, exist_ok=True)

FINAL_PREDICTIONS29 = (
    ROOT29 / "cross_encoder_predictions_480.jsonl"
)

FINAL_MANIFEST29 = (
    ROOT29 / "cross_encoder_predictions_manifest.json"
)

# --------------------------------------------------
# 2. Helpers
# --------------------------------------------------

def sha29(data):
    return hashlib.sha256(data).hexdigest()


def read_jsonl29(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


def json_bytes29(record):
    return (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True,
            allow_nan=False,
        ) + "\n"
    ).encode("utf-8")


def save_safely29(path, data):

    if path.exists():
        assert path.read_bytes() == data, (
            f"Existing file differs: {path}"
        )
    else:
        temp = path.with_name(path.name + ".tmp")
        temp.write_bytes(data)
        temp.replace(path)

    assert path.read_bytes() == data


# --------------------------------------------------
# 3. Verify active model and frozen protocol
# --------------------------------------------------

required29 = [
    "reranker09",
    "doc_by_id09",
    "RERANK_MODEL_ID09",
    "RERANK_REVISION09",
    "RERANK_MAX_LENGTH09",
    "RERANK_BATCH_SIZE09",
    "RERANK_DEVICE09",
]

missing29 = [
    name for name in required29
    if name not in globals()
]

assert not missing29, (
    "Missing runtime variables: " + ", ".join(missing29)
)

assert RERANK_DEVICE09 == "cpu"
assert RERANK_MAX_LENGTH09 == 512
assert len(doc_by_id09) == 5250
assert MAX_NEW_QUERIES29 > 0

protocol_bytes29 = PROTOCOL_FILE29.read_bytes()
setup_bytes29 = SETUP_FILE29.read_bytes()
query_bytes29 = QUERY_FILE29.read_bytes()
candidate_bytes29 = CANDIDATE_FILE29.read_bytes()

protocol29 = json.loads(protocol_bytes29)
setup29 = json.loads(setup_bytes29)

candidate_manifest29 = json.loads(
    CANDIDATE_MANIFEST29.read_text(encoding="utf-8")
)

protocol_sha29 = sha29(protocol_bytes29)
setup_sha29 = sha29(setup_bytes29)
query_sha29 = sha29(query_bytes29)
candidate_sha29 = sha29(candidate_bytes29)

assert setup29["protocol_sha256"] == protocol_sha29
assert setup29["frozen_queries_sha256"] == query_sha29

assert candidate_manifest29["status"] == "complete"
assert candidate_manifest29["n_queries"] == 480

assert (
    candidate_manifest29["candidate_file_sha256"]
    == candidate_sha29
)

assert (
    candidate_manifest29["protocol_sha256"]
    == protocol_sha29
)

assert (
    candidate_manifest29["frozen_queries_sha256"]
    == query_sha29
)

assert protocol29["model"]["id"] == RERANK_MODEL_ID09
assert protocol29["model"]["revision"] == RERANK_REVISION09
assert protocol29["model"]["max_length"] == 512
assert protocol29["model"]["batch_size"] == RERANK_BATCH_SIZE09

assert protocol29["retrieval"]["rerank_top_k"] == 5

print("CELL 29 — AUTOMATIC CROSS-ENCODER EVALUATION")
print("=" * 60)

print("Frozen protocol: VERIFIED")
print("Candidate file SHA256: VERIFIED")
print("Cross-Encoder model: VERIFIED")
print("Rerank Top-K: 5")
print("Device: CPU")

# --------------------------------------------------
# 4. Load and verify 480 evaluation inputs
# --------------------------------------------------

queries29 = read_jsonl29(QUERY_FILE29)
candidate_pools29 = read_jsonl29(CANDIDATE_FILE29)

assert len(queries29) == 480
assert len(candidate_pools29) == 480

assert len({
    q["query_id"] for q in queries29
}) == 480

for q, pool in zip(queries29, candidate_pools29):

    assert q["query_id"] == pool["query_id"]
    assert q["query"] == pool["query"]
    assert q["query_type"] == pool["query_type"]

    assert q["source_id"] == pool["source_id"]

    assert pool["partition"] == "evaluation"

    ids = [
        c["document_id"]
        for c in pool["candidates"]
    ]

    assert 5 <= len(ids) <= 100
    assert len(ids) == len(set(ids))

    assert all(
        doc_id in doc_by_id09
        for doc_id in ids
    )

# --------------------------------------------------
# 5. Checkpoint verification
# --------------------------------------------------

def checkpoint_path29(pool):
    return (
        CHECKPOINT_DIR29
        / f"{pool['query_id']}.json"
    )


def prepare_inputs29(pool):

    top5 = pool["candidates"][:5]

    pairs = [
        (
            pool["query"],
            doc_by_id09[c["document_id"]]["text"]
        )
        for c in top5
    ]

    # Fingerprint the actual query-document inputs.
    input_digest = sha29(
        json_bytes29({
            "query": pool["query"],
            "documents": [
                {
                    "document_id": c["document_id"],
                    "text_sha256": sha29(
                        doc_by_id09[
                            c["document_id"]
                        ]["text"].encode("utf-8")
                    ),
                }
                for c in top5
            ],
        })
    )

    return top5, pairs, input_digest


def verify_checkpoint29(pool, record):

    top5, _, input_digest = prepare_inputs29(pool)

    original_ids = [
        c["document_id"]
        for c in top5
    ]

    assert record["query_id"] == pool["query_id"]
    assert record["query_type"] == pool["query_type"]

    assert record["protocol_sha256"] == protocol_sha29
    assert record["candidate_sha256"] == candidate_sha29

    assert record["model_id"] == RERANK_MODEL_ID09
    assert record["model_revision"] == RERANK_REVISION09

    assert record["rerank_top_k"] == 5
    assert record["max_length"] == 512
    assert record["input_sha256"] == input_digest

    assert record["hybrid_top5_ids"] == original_ids

    ranked = record["reranked_top5"]

    assert len(ranked) == 5

    assert {
        r["document_id"] for r in ranked
    } == set(original_ids)

    assert all(
        np.isfinite(r["reranker_score"])
        for r in ranked
    )

    assert ranked == sorted(
        ranked,
        key=lambda r: (
            -r["reranker_score"],
            r["hybrid_rank"],
            r["document_id"],
        )
    )


# --------------------------------------------------
# 6. Find pending queries
# --------------------------------------------------

pending29 = []

for pool in candidate_pools29:

    path = checkpoint_path29(pool)

    if path.is_file():

        record = json.loads(
            path.read_text(encoding="utf-8")
        )

        verify_checkpoint29(pool, record)

    else:

        pending29.append(pool)

print("\nEVALUATION PROGRESS")
print("-" * 55)

print(f"Completed: {480 - len(pending29)}/480")
print(f"Remaining: {len(pending29)}")
print(f"Batch size: {MAX_NEW_QUERIES29}")

# --------------------------------------------------
# 7. Run next pending batch
# --------------------------------------------------

selected29 = pending29[:MAX_NEW_QUERIES29]

batch_start29 = time.perf_counter()

for index, pool in enumerate(selected29, start=1):

    qid = pool["query_id"]

    top5, pairs, input_digest = prepare_inputs29(pool)

    print(
        f"[{index}/{len(selected29)}] "
        f"Running {qid}...",
        flush=True
    )

    start = time.perf_counter()

    scores = reranker09.predict(
        pairs,
        batch_size=RERANK_BATCH_SIZE09,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    elapsed = time.perf_counter() - start

    scores = np.asarray(
        scores,
        dtype=float
    ).reshape(-1)

    assert scores.shape == (5,)
    assert np.isfinite(scores).all()

    ranked = [
        {
            "document_id": candidate["document_id"],
            "hybrid_rank": int(candidate["rank"]),
            "reranker_score": float(score),
        }
        for candidate, score in zip(top5, scores)
    ]

    ranked.sort(
        key=lambda r: (
            -r["reranker_score"],
            r["hybrid_rank"],
            r["document_id"],
        )
    )

    record = {
        "query_id": qid,
        "query_type": pool["query_type"],
        "partition": "evaluation",
        "model_id": RERANK_MODEL_ID09,
        "model_revision": RERANK_REVISION09,
        "protocol_sha256": protocol_sha29,
        "candidate_sha256": candidate_sha29,
        "input_sha256": input_digest,
        "rerank_top_k": 5,
        "max_length": 512,
        "batch_size": RERANK_BATCH_SIZE09,
        "hybrid_top5_ids": [
            c["document_id"] for c in top5
        ],
        "reranked_top5": ranked,
        "inference_seconds": float(elapsed),
    }

    verify_checkpoint29(pool, record)

    path = checkpoint_path29(pool)

    save_safely29(
        path,
        json_bytes29(record)
    )

    print(
        f"  {elapsed:.2f} sec | SAVED",
        flush=True
    )

# --------------------------------------------------
# 8. Calculate progress — not evaluation metrics
# --------------------------------------------------

remaining29 = [
    pool
    for pool in candidate_pools29
    if not checkpoint_path29(pool).is_file()
]

completed29 = 480 - len(remaining29)

batch_elapsed29 = (
    time.perf_counter() - batch_start29
)

print("\n" + "=" * 60)
print("EVALUATION INFERENCE PROGRESS")
print("=" * 60)

print(f"Completed: {completed29}/480")
print(f"Remaining: {len(remaining29)}")

print(
    "Current batch elapsed:",
    f"{batch_elapsed29 / 60:.2f} minutes"
)

# --------------------------------------------------
# 9. Finalize predictions only at 480/480
# --------------------------------------------------

if not remaining29:

    all_predictions29 = []

    for pool in candidate_pools29:

        record = json.loads(
            checkpoint_path29(pool).read_text(
                encoding="utf-8"
            )
        )

        verify_checkpoint29(pool, record)

        all_predictions29.append(record)

    predictions_bytes29 = b"".join(
        json_bytes29(record)
        for record in all_predictions29
    )

    predictions_sha29 = sha29(
        predictions_bytes29
    )

    save_safely29(
        FINAL_PREDICTIONS29,
        predictions_bytes29
    )

    prediction_manifest29 = {
        "experiment": "cross_encoder_frozen_eval_v1",
        "status": "inference_complete",
        "n_queries": 480,
        "model": protocol29["model"],
        "retrieval": protocol29["retrieval"],
        "protocol_sha256": protocol_sha29,
        "evaluation_setup_sha256": setup_sha29,
        "frozen_queries_sha256": query_sha29,
        "candidate_sha256": candidate_sha29,
        "predictions_file": FINAL_PREDICTIONS29.name,
        "predictions_sha256": predictions_sha29,
        "evaluation_metrics_computed": False,
    }

    save_safely29(
        FINAL_MANIFEST29,
        json_bytes29(prediction_manifest29)
    )

    print("\nALL 480 EVALUATION QUERIES COMPLETED!")

    print("\nPredictions SHA256:")
    print(predictions_sha29)

    print("\nOK — Cross-Encoder evaluation inference completed")

else:

    print("\nRun this SAME Cell 29 again to continue.")
    print("Saved checkpoints will be reused.")

print("\nEvaluation metrics: NOT COMPUTED")
print("Model training: NOT PERFORMED")
print("Frozen protocol: UNCHANGED")